# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 4/12** (CPU, internet off). **1778 series**, ~10.6 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part4.zip`
- `manifest_rgb25d_320_part4.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 4
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260","series":"1.2.826.0.1.3680043.8.498.75714899997203615784077798038670363546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.10997243274493644695137773062377650204","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.96277058087102759370159028042426256271","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.78517915125227969420323795995142127096","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394","series":"1.2.826.0.1.3680043.8.498.10039810104099273075015998627058683073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981","series":"1.2.826.0.1.3680043.8.498.10261053202170116880341199746093649009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866","series":"1.2.826.0.1.3680043.8.498.80126107048478277295386095863259178495","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269","series":"1.2.826.0.1.3680043.8.498.21392572370649076486729381864268170044","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736","series":"1.2.826.0.1.3680043.8.498.52609252494549544989037353476227811522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941","series":"1.2.826.0.1.3680043.8.498.11937405551646049903918830246970006283","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.12943110519784474302391125333965928911","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","series":"1.2.826.0.1.3680043.8.498.19623053198315082370818305051668483810","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863","series":"1.2.826.0.1.3680043.8.498.91348587995236577129146264967955930115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.10806209525568981870436366736339073203","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.12057161078941881649917424421113939485","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339","series":"1.2.826.0.1.3680043.8.498.11045479064503184057854236314537486044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833","series":"1.2.826.0.1.3680043.8.498.84628930131103867630611531942579770333","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166","series":"1.2.826.0.1.3680043.8.498.84878015408307681578702917309931549382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.83928691140493975131372679311192757705","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.61177836839458588461273706533138857053","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.89200806589520831313379005384201520295","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019","series":"1.2.826.0.1.3680043.8.498.61178174888593430207613886253463045477","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.71632516016159079115119797092655911971","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484","series":"1.2.826.0.1.3680043.8.498.23665757498452344201495732325817484986","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.26549824353145138262302671279541386781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.25511961362790207336672525814885780079","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.27751934179631865487758456365935105086","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292","series":"1.2.826.0.1.3680043.8.498.84809656552963955311538050630463612106","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039","series":"1.2.826.0.1.3680043.8.498.10311881159524822194371683400431695124","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773","series":"1.2.826.0.1.3680043.8.498.59521695827464303188269281692322338337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183","series":"1.2.826.0.1.3680043.8.498.52417591829781666982794713829779567002","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585","series":"1.2.826.0.1.3680043.8.498.11529309520105279537501270800913260335","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.74012623672629253241754830905879289073","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.10908568304623504035467450279262604082","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326","series":"1.2.826.0.1.3680043.8.498.84420106226634229715724242726940331787","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307","series":"1.2.826.0.1.3680043.8.498.76971352601713217956399819344758048274","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.10516784608287332404311877663610757299","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.58424081772009549276617433634553316660","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903","series":"1.2.826.0.1.3680043.8.498.39247763432095298492907510497938662680","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561","series":"1.2.826.0.1.3680043.8.498.11887128415566066538639811393074744572","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029","series":"1.2.826.0.1.3680043.8.498.10317127485116890640697878607599865986","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077","series":"1.2.826.0.1.3680043.8.498.13348065917387117767990304461512778701","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669","series":"1.2.826.0.1.3680043.8.498.54137868922970612269408471390702543246","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097","series":"1.2.826.0.1.3680043.8.498.14894618540297979515666221458834864736","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052","series":"1.2.826.0.1.3680043.8.498.55427564251500418323672579441288025575","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.20218513297383810986327603522993399171","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725","series":"1.2.826.0.1.3680043.8.498.97011119464988985971367876859337476583","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952","series":"1.2.826.0.1.3680043.8.498.60756735624203738766257503911815676720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.82357545864671391890323608699227577591","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687","series":"1.2.826.0.1.3680043.8.498.24273324148338116002145163684564899967","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753","series":"1.2.826.0.1.3680043.8.498.11239581556011071804203065202198795958","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306","series":"1.2.826.0.1.3680043.8.498.67629398946638919507118338970024586329","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.11516970453859080724649683058969205867","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.19804024856465410145129691722550712124","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247","series":"1.2.826.0.1.3680043.8.498.46635992881064097163699454530900405525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700","series":"1.2.826.0.1.3680043.8.498.91573541961438772021922050933517826694","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.98281342676472765586584090313024669293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.40149591164211868964032347604988945819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933","series":"1.2.826.0.1.3680043.8.498.68027724471167073605029503597769156815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334","series":"1.2.826.0.1.3680043.8.498.27740666011617851013638431808469377511","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434","series":"1.2.826.0.1.3680043.8.498.46497786100798823729654783386518899025","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715","series":"1.2.826.0.1.3680043.8.498.43071561606195363007720017904622333043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","series":"1.2.826.0.1.3680043.8.498.87239186827932355007728827241327160851","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626","series":"1.2.826.0.1.3680043.8.498.48139147829736277873756004699021511315","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056","series":"1.2.826.0.1.3680043.8.498.77212768576945603832682052942029386696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561","series":"1.2.826.0.1.3680043.8.498.12136046570218320617848301998799159537","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.27863777403538335221468543729558746103","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397","series":"1.2.826.0.1.3680043.8.498.12884867359658923600814195849320923942","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.63619777164345825683505292291360466768","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.40258877787887134433688592772246515182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326","series":"1.2.826.0.1.3680043.8.498.10448868261060659272030211072291778496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584","series":"1.2.826.0.1.3680043.8.498.60937579538433294438916040161180815564","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.73379701973537794457450903519717734057","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.62019145825711385539085785610410456612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213","series":"1.2.826.0.1.3680043.8.498.12458602904400384303589710829232464192","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015","series":"1.2.826.0.1.3680043.8.498.10089454533689575892159684540414117845","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.11752312521368041265933049669258908763","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739","series":"1.2.826.0.1.3680043.8.498.62407820280040385144206393521553225916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.12943328872774722338657758334379109857","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573","series":"1.2.826.0.1.3680043.8.498.84936110136857195185463602259491980002","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.55893467756430372796475235957862198880","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251","series":"1.2.826.0.1.3680043.8.498.68069268021270436167574707681412656223","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.48493369876249548834628261596773392489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109","series":"1.2.826.0.1.3680043.8.498.11672754659958062462850109130282817404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.37385495163062009601462071498837729650","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10589889055723532336864750992142411145","series":"1.2.826.0.1.3680043.8.498.19456599046232988845616700737431591582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916","series":"1.2.826.0.1.3680043.8.498.41261876638659855638885924104976797096","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137","series":"1.2.826.0.1.3680043.8.498.10457722103437597353351370290626803648","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.71578918055169027874832428784102882394","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.77879915181981128945402062453846214311","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403","series":"1.2.826.0.1.3680043.8.498.34086045337446070070158657434276511823","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839","series":"1.2.826.0.1.3680043.8.498.72136322538200737638335322212985749652","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912","series":"1.2.826.0.1.3680043.8.498.26340552236924824314951040308700787497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477","series":"1.2.826.0.1.3680043.8.498.24872275347880480636927394543888122406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.10406297933531544965301320719736404252","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374","series":"1.2.826.0.1.3680043.8.498.69576913252372448353694338511725368873","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967","series":"1.2.826.0.1.3680043.8.498.98819086222791903928887402715173374973","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.48946866567389310566625166497195768307","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.23129658291260452515416514750613208894","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731","series":"1.2.826.0.1.3680043.8.498.23934316670432021861006355081092376905","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.10644662943826161045820718738445502175","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.59864558861260857063358712480488082714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654","series":"1.2.826.0.1.3680043.8.498.12148085871521049340320769822265487887","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.23348762085643529567013843212388008216","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879","series":"1.2.826.0.1.3680043.8.498.22273540801584220403872600700256675655","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557","series":"1.2.826.0.1.3680043.8.498.13771001492454642170755478476032403803","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384","series":"1.2.826.0.1.3680043.8.498.12051847207401924553180919895524463827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.15152833144727504785224591596920173845","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093","series":"1.2.826.0.1.3680043.8.498.80578660557040452814256943827961916646","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327","series":"1.2.826.0.1.3680043.8.498.15949238077902120011774507698057228425","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826","series":"1.2.826.0.1.3680043.8.498.37194922478966602983662524585374540365","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762","series":"1.2.826.0.1.3680043.8.498.88613225461250490619871769447186164572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448","series":"1.2.826.0.1.3680043.8.498.56357744673441831360063211839627398666","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555","series":"1.2.826.0.1.3680043.8.498.35391566097837852046533210559703441829","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770","series":"1.2.826.0.1.3680043.8.498.82986046279533956313630293638852021340","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.12331790798775573019772875063818723347","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406","series":"1.2.826.0.1.3680043.8.498.11167320851479885907988585803482877590","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.46389127916509006601593709659772613495","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738","series":"1.2.826.0.1.3680043.8.498.82593276553567218720270019902604201028","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263","series":"1.2.826.0.1.3680043.8.498.70812696834097098874616664473334967936","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650","series":"1.2.826.0.1.3680043.8.498.40584409397736404276648715574734534299","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636","series":"1.2.826.0.1.3680043.8.498.92153985350692437713069670214045351658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766","series":"1.2.826.0.1.3680043.8.498.75116070091968307886032230843296723873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.67272569719608559014149588716882496149","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385","series":"1.2.826.0.1.3680043.8.498.45580399222613365402730380613056139221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748","series":"1.2.826.0.1.3680043.8.498.89837873571047667182183544654990579046","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834","series":"1.2.826.0.1.3680043.8.498.11835714984191741802606652870152504726","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412","series":"1.2.826.0.1.3680043.8.498.23298980182136338223886053595997861044","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.46293903826829324792025836895014796249","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235","series":"1.2.826.0.1.3680043.8.498.67922870009354031704812995034985923566","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566","series":"1.2.826.0.1.3680043.8.498.11536806626719859744100015982574383734","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403","series":"1.2.826.0.1.3680043.8.498.57102998217286118624861770119728259811","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658","series":"1.2.826.0.1.3680043.8.498.71958650339512981041833071212420888299","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606","series":"1.2.826.0.1.3680043.8.498.16660104770503781152156668339060998752","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.11092117420899066470095241248229627325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.94033065213746557708526442135075463051","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245","series":"1.2.826.0.1.3680043.8.498.12270167051631451660773334136521239558","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106","series":"1.2.826.0.1.3680043.8.498.63373874637657760702752299660526643003","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962","series":"1.2.826.0.1.3680043.8.498.11834523744270534261277155527481275830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445","series":"1.2.826.0.1.3680043.8.498.12654975596045953747404323534801495735","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979","series":"1.2.826.0.1.3680043.8.498.13256034505713854501750467470076108342","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054","series":"1.2.826.0.1.3680043.8.498.10140887816944112044742717004984233349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.12152336407945042070903349786041063784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11011061033863592500429481719734134286","series":"1.2.826.0.1.3680043.8.498.59212444311121427152789813716263135767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.12556318568424369818960973936088155711","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.10066611045150267821217000314477504439","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11049882927267755769995376553330493298","series":"1.2.826.0.1.3680043.8.498.10606822046928869563426838025941558340","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739","series":"1.2.826.0.1.3680043.8.498.11292064626878451339228234335926212863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.41220134823084008278714063897444422461","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11070900122260301212131296598983372089","series":"1.2.826.0.1.3680043.8.498.54453466763416462206996335236143653002","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445","series":"1.2.826.0.1.3680043.8.498.79575780777096540288203277186570180412","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11090884548990390279582832205294943635","series":"1.2.826.0.1.3680043.8.498.33924476870655602036880022285772217520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.67343214684927208514011163481472891900","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11111022632180183697497694737350901427","series":"1.2.826.0.1.3680043.8.498.60001038797219155864440533893501452796","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.21936721604801160847259894235727804008","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.96205657150098886651447434041345939522","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11128099896284546522122813576483001027","series":"1.2.826.0.1.3680043.8.498.59310271184457629925168246975075209030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.61186952500689646697728005450647570661","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.31654690739865940043199846757467298133","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11149530279799223497509928655010867960","series":"1.2.826.0.1.3680043.8.498.96272498477370746132605408584462101679","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150597567791052354257165699079348456","series":"1.2.826.0.1.3680043.8.498.12322997316383985816050208157048115259","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605","series":"1.2.826.0.1.3680043.8.498.32638787614667845841425837662438423769","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11159546221032123495135103648570690656","series":"1.2.826.0.1.3680043.8.498.87386226856144144002989951755158731329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.65658586178897676044342148845821358632","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11190292536746235063218740991532759863","series":"1.2.826.0.1.3680043.8.498.69021611747964957472966802209549674377","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618","series":"1.2.826.0.1.3680043.8.498.12641742550562840625727412606175814415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824","series":"1.2.826.0.1.3680043.8.498.10937670737582286997380405965339084017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11208663314213048426907645330143931419","series":"1.2.826.0.1.3680043.8.498.11614932689158505592882809186497438228","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11212667888429022863778740306223585731","series":"1.2.826.0.1.3680043.8.498.30364471691161649690685534256178507785","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678","series":"1.2.826.0.1.3680043.8.498.66536508229204139448661585580659332151","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11221318524515817857172161854667955115","series":"1.2.826.0.1.3680043.8.498.46679481793737847799943060792534576696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.10972653201273094675585632762550406091","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.93827564998232296482906708217262150323","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11249981834802444119817546155435759007","series":"1.2.826.0.1.3680043.8.498.23261656997277612738589315283577262170","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.12237982259143054202327200840572040586","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11262944211104069231500934582412003273","series":"1.2.826.0.1.3680043.8.498.28769030233927332109579624359642442703","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.39063003742062762578153861907339118087","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11276674006815021828943434205349220551","series":"1.2.826.0.1.3680043.8.498.12606790664427607668261541378558920731","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11277278232377185347750793496260885546","series":"1.2.826.0.1.3680043.8.498.11156458226846453891094295258008626100","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282706861378379965012380644125662529","series":"1.2.826.0.1.3680043.8.498.37671375170361167645864598416838388843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.55873417497300701849234184809457061290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11295899132697862632905938037090529644","series":"1.2.826.0.1.3680043.8.498.74800672226731313054975198837566895537","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.12328781530272644444391978821475342444","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.10263393409962371896663989525025400861","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055","series":"1.2.826.0.1.3680043.8.498.76484099929061872305137206391285382145","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.80420630865395058127857830708142911797","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11337616353935430160375524217870968575","series":"1.2.826.0.1.3680043.8.498.88835535559101267374596312311000548280","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","series":"1.2.826.0.1.3680043.8.498.65234988957425994870824967664983745789","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","series":"1.2.826.0.1.3680043.8.498.13030837831142984863298092731656186977","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.21563620570271797548468978513260466177","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238","series":"1.2.826.0.1.3680043.8.498.38450243001981446118642153746249112051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899","series":"1.2.826.0.1.3680043.8.498.18346563263680985907823070140144466133","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","series":"1.2.826.0.1.3680043.8.498.79152502003885292626651018378568018768","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11402325536521359105285323485008939602","series":"1.2.826.0.1.3680043.8.498.61725663844680992053964431506647935219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11413427362266862411525989114324631512","series":"1.2.826.0.1.3680043.8.498.55184705690351013779833454322146558040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.81208577373030268081921068298070045708","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.27873771543687985144892825453784598580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440214477653629899108986507154583794","series":"1.2.826.0.1.3680043.8.498.45901428374073786927873108127500246829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.11108527560863392333572742157671075816","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11448865770537436004548718998804636234","series":"1.2.826.0.1.3680043.8.498.23854356002166046342050769882483902981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.21615027520865311148040358637280926279","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.15535676340856192556666660993915309674","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11467955500991327547315970867313333592","series":"1.2.826.0.1.3680043.8.498.31840433025096098808733103347248551915","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.21157261361701207741689780022259795713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.81658576155023183858034500312979561441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11489881286208501558797333161632486099","series":"1.2.826.0.1.3680043.8.498.75180777891047733416419172126891074074","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11497761063752461132736045914751707046","series":"1.2.826.0.1.3680043.8.498.11554491089530578506631036182858440062","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.12163675020152080514130880976560545760","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588","series":"1.2.826.0.1.3680043.8.498.70674724726413128502777455189020011759","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.10562674363675629697402467833740271244","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.10638794715372528088627202144595867479","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11520508017843738484108845144418061602","series":"1.2.826.0.1.3680043.8.498.10537905482904599135655877222272214934","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","series":"1.2.826.0.1.3680043.8.498.44979370804815230109313784202458403327","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.39378460513013835352482765355326997812","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11545541207342029419461179316053148974","series":"1.2.826.0.1.3680043.8.498.27480308200491154173139300637720892392","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.11483777317678061466864607463133892766","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11551347885357242022697196552822735795","series":"1.2.826.0.1.3680043.8.498.73011146746167275493651665388503262561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.56343388006538255668045138552517984731","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11569601172991867969672947467467392425","series":"1.2.826.0.1.3680043.8.498.10013168686774212100388186060914556274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.13141571267133221745510704379010051774","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610","series":"1.2.826.0.1.3680043.8.498.10621746995288865071412920860014823631","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11586616391541918134110642641868079265","series":"1.2.826.0.1.3680043.8.498.56184805590447254047835126869599803011","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.71897454394240427591302309329142928097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.93670395223792512293996254829085321413","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11596866241403598097171345671773992202","series":"1.2.826.0.1.3680043.8.498.63714359393851835865126331689434668921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511","series":"1.2.826.0.1.3680043.8.498.83257280326369198056295498765063962040","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.73216197222483630190647913351304388691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11613637647831801354398273334552399874","series":"1.2.826.0.1.3680043.8.498.12859489622346080768066017006825887675","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097","series":"1.2.826.0.1.3680043.8.498.12570252388561688704180692611643431319","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.88812267725892788336897993419718778579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11642678020345468499556230871411308014","series":"1.2.826.0.1.3680043.8.498.12484053255624940250744701796050154312","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853","series":"1.2.826.0.1.3680043.8.498.20757775547689205966982402990561680580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","series":"1.2.826.0.1.3680043.8.498.66733268482077391033469514354090103790","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.10057305864227595980423089397727389218","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11674925658184069242322832524274428408","series":"1.2.826.0.1.3680043.8.498.10631379905248886281584110672360292337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.83034037641734450392520881538349364263","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11689820638581431730501617718741779835","series":"1.2.826.0.1.3680043.8.498.22538147688575476629567878400808353223","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11695097367686121784104505410611821860","series":"1.2.826.0.1.3680043.8.498.54727530624937855904524209414900516908","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256","series":"1.2.826.0.1.3680043.8.498.97250292486177322506079559223772623346","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","series":"1.2.826.0.1.3680043.8.498.82308380197687562468418335529158712949","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.55787743694464420631976381966219903287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723286647024326363349453996818420374","series":"1.2.826.0.1.3680043.8.498.11466631424956940350107476245680565815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11726452645519338341002251323463736083","series":"1.2.826.0.1.3680043.8.498.11963199713436856539956893611255293285","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225","series":"1.2.826.0.1.3680043.8.498.10217650002674853227555594155252659571","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025","series":"1.2.826.0.1.3680043.8.498.45097510702863003175133082599107849201","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.11879842328445957013794795008704916170","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.78542777773195468248339797405014654540","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.19833365147722679314138735776189211029","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.38596983255460050384330303364267931918","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.17170538579649104376355022891111930515","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.12818400887641328623268848607800260625","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11804799183537166337413388415687675758","series":"1.2.826.0.1.3680043.8.498.13164396606081500256192268954140462706","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.68745329426797900119745170516653245129","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.51895633154663765942890035716151589024","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11836929166973549727078374306175992474","series":"1.2.826.0.1.3680043.8.498.57374985740985872350719251839877660233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.38946384864117135171384663677960789880","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129","series":"1.2.826.0.1.3680043.8.498.22460211714375534713354977697684334277","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11868389442143514705330687746274512632","series":"1.2.826.0.1.3680043.8.498.10498481083444930147136311522845701564","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11878847808072119549790690649256090471","series":"1.2.826.0.1.3680043.8.498.14666079282551054617472153640359581994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.10497958774005576258456056141881811578","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11897083656099489353496670273111097794","series":"1.2.826.0.1.3680043.8.498.11492946892066555765464590760279522708","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861","series":"1.2.826.0.1.3680043.8.498.40581200863932491307529183756256354078","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734","series":"1.2.826.0.1.3680043.8.498.49736684438569190684699777814575056960","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","series":"1.2.826.0.1.3680043.8.498.28044433299531804966246977866753733475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11915933561045570822185908784072057715","series":"1.2.826.0.1.3680043.8.498.85262385357980228699742034305097551865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.14731039237922793213007066391644414535","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920","series":"1.2.826.0.1.3680043.8.498.13308545092523025146507120149148250355","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11927352926669104658144933891780551483","series":"1.2.826.0.1.3680043.8.498.12170508358775324589702715935884393497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770","series":"1.2.826.0.1.3680043.8.498.12989172756167506158170308173292158963","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11938864543152001817866013174907274014","series":"1.2.826.0.1.3680043.8.498.10851217338035309822470765102634698843","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","series":"1.2.826.0.1.3680043.8.498.30404680786132269406467263369608679694","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.15889566039883165067353236284952567659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.70094218287485178339400861726474313405","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443","series":"1.2.826.0.1.3680043.8.498.12545014067680078602976025863351853237","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.48577986385478681225329495686830738076","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.71074591215970878844614053209922285344","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12005327370242618323737652892230673473","series":"1.2.826.0.1.3680043.8.498.12556033271812146203169386429759673696","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723","series":"1.2.826.0.1.3680043.8.498.73069660406894528786763755352421104272","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12014959202771141042815134335571931814","series":"1.2.826.0.1.3680043.8.498.56199137416899181727841118732266910369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12026457191304650188547548517205757122","series":"1.2.826.0.1.3680043.8.498.74614829868048968708574793698665879077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.81712637977161795523558008311300041525","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.10049652568021778646861430531680266367","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813","series":"1.2.826.0.1.3680043.8.498.10897498939961960060408429924316224507","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12063742398421783626420335095128808816","series":"1.2.826.0.1.3680043.8.498.10966318154445739523966073417426318408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.19187956989158280679911518782944510534","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12073401429398042797505820525732905336","series":"1.2.826.0.1.3680043.8.498.33106221486778596583987511197044205525","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","series":"1.2.826.0.1.3680043.8.498.68144028652813687090226707330259206915","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12104799278874963688461235330096856415","series":"1.2.826.0.1.3680043.8.498.44404077948663198679914776006039316295","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12110510829608190094905588610131531103","series":"1.2.826.0.1.3680043.8.498.78600544684275475318510946289951878793","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761","series":"1.2.826.0.1.3680043.8.498.57250678761408366098875733912347695362","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.22971550578791233452375899870035021863","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12139829070290803654867114374799823711","series":"1.2.826.0.1.3680043.8.498.12272992332570007168759508856785759999","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.58028941983023213489384057349775249830","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12158630368692962312115489365152319457","series":"1.2.826.0.1.3680043.8.498.83056656597690907204422999839709990053","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.17761596020888930733962796808823049736","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12168045829469202472124733625621943469","series":"1.2.826.0.1.3680043.8.498.91283996788859384654547158187372776901","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.60389734547526291840929384108242438222","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044","series":"1.2.826.0.1.3680043.8.498.10646717378564079924131308380153257139","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12181591358572249491885248089245858515","series":"1.2.826.0.1.3680043.8.498.46576951333789481560630987482847239643","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.44195921250242998302179423736796357537","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939","series":"1.2.826.0.1.3680043.8.498.32562983956115855488691329373720169023","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12200672323103829956274910258854933266","series":"1.2.826.0.1.3680043.8.498.11498720533774178571608478355214348583","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.10336405823401566804045315181443406949","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12215902620611618183321984433114460986","series":"1.2.826.0.1.3680043.8.498.13292860333213472735440496459601236832","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.10881884503767324720013764292658462485","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988","series":"1.2.826.0.1.3680043.8.498.43751510994750683203085623396976308546","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12238865510201989224648662550299925120","series":"1.2.826.0.1.3680043.8.498.15263926101118141564543111793059869091","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12242172267695588655664456078003635837","series":"1.2.826.0.1.3680043.8.498.71768483873778980204482305312786949840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.62664900114544925434949878764949844024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.89858881895612879602646374501929112752","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12257484608743859456600273083454447341","series":"1.2.826.0.1.3680043.8.498.12805997643886622954031598585855575764","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336","series":"1.2.826.0.1.3680043.8.498.62036778818246560582989341221470454972","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.99381667403478945732702542019152014667","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12267944258766605501055560486655358039","series":"1.2.826.0.1.3680043.8.498.51687232647034974061178306169512946404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12273356591019341319494035708698721367","series":"1.2.826.0.1.3680043.8.498.84161660503593910931086196052556602929","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406","series":"1.2.826.0.1.3680043.8.498.11273932285011210306123557912905884297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.10948861780422422612998003064849798696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734","series":"1.2.826.0.1.3680043.8.498.49683639839757231075304295273451019862","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.60801076221815362135650562144071018356","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.47800591297556363048589135347028060855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12301945990161670321760708884248055936","series":"1.2.826.0.1.3680043.8.498.68372011121335804423897993098878065486","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573","series":"1.2.826.0.1.3680043.8.498.12283736734531453211089656474137674206","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.10830137247626538802460343146253900053","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.90278443346626867350765162900005017158","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151","series":"1.2.826.0.1.3680043.8.498.35481743181880561929207574028267471198","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12335573784261147388984372283481919933","series":"1.2.826.0.1.3680043.8.498.42760722228653623940546377534359870204","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","series":"1.2.826.0.1.3680043.8.498.12574165175410256113967869386216543721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348960754791091583201828167607825033","series":"1.2.826.0.1.3680043.8.498.11679274850421961554985348088691763758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648","series":"1.2.826.0.1.3680043.8.498.50972061382834950795125071184197246102","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12356611369549236159671896247094661767","series":"1.2.826.0.1.3680043.8.498.57190103821310215773100073637699681774","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973","series":"1.2.826.0.1.3680043.8.498.10759970126679580232390903884038907793","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785","series":"1.2.826.0.1.3680043.8.498.24380565001952303917864952593178583464","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.73859996272463942305463736605930037773","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12383716855639375235119728470042467411","series":"1.2.826.0.1.3680043.8.498.60593596220273975925029217159774337400","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.69226923006017947075905253867729173579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12390186846317493876612228407734068726","series":"1.2.826.0.1.3680043.8.498.22073051704842220310192829301026702569","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.10608285129088648270877133835763781380","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.43501173635498296671950614403286208443","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12411340904876460326022004747253138831","series":"1.2.826.0.1.3680043.8.498.12479092981006149959962827802064941875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12426381920136706157912507136730458010","series":"1.2.826.0.1.3680043.8.498.66984502376609073384928787720502458078","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","series":"1.2.826.0.1.3680043.8.498.10710912551305115439635403049715101778","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.52417526066591351366797282021650019781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12450858400030959723767712312663176092","series":"1.2.826.0.1.3680043.8.498.10817646574412003191457281490157916784","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12462703906850759490077163612851243080","series":"1.2.826.0.1.3680043.8.498.11978315170964233536520130318258876503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949","series":"1.2.826.0.1.3680043.8.498.96371873040537284572175608603291741446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12482482491088588051076210158062455084","series":"1.2.826.0.1.3680043.8.498.14852634347378027776462319542357336061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.22204020867787235387913012086919880209","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.41117743871226461051047060035449157254","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.11297549143137064452089015034845808715","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12497144849433082154223897356471317072","series":"1.2.826.0.1.3680043.8.498.74838435207223961421032431845276026916","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819","series":"1.2.826.0.1.3680043.8.498.96225261274596688102834905835286265015","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12509825237087140646629745447753158673","series":"1.2.826.0.1.3680043.8.498.93408627930291627087023617912128870500","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592","series":"1.2.826.0.1.3680043.8.498.34299474734610452533702929263408256310","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.56702793783211669331707923230912261989","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12523676732876902778577970655429060858","series":"1.2.826.0.1.3680043.8.498.13456530769614259822586462346402896198","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.45615489149115811625013810587139154605","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.82671709578855608637080736868257249579","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12547308734950397389859712060653254111","series":"1.2.826.0.1.3680043.8.498.10388167112612514969774053164125712764","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826","series":"1.2.826.0.1.3680043.8.498.13857708683711931762348466566381533641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12554824497579383319980187351290254727","series":"1.2.826.0.1.3680043.8.498.15805697535419860634147746088665840844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.44366857457846866559851276235391742174","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12570576337098834311171883034999327454","series":"1.2.826.0.1.3680043.8.498.12996239004829249660659265036361035753","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12574270269124650675934073761741661292","series":"1.2.826.0.1.3680043.8.498.13434230692165812727012540776257380073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","series":"1.2.826.0.1.3680043.8.498.12333374080887330762304608152978379492","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.79082476308244383822509323373638035695","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.36400594315548091590932574077827580913","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607659092357834912029375690348886159","series":"1.2.826.0.1.3680043.8.498.75094499907214968747718308448592898104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.86510927001798876186444024797559879357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12620718517255799151283445360332799702","series":"1.2.826.0.1.3680043.8.498.43435796116077822840655245497967980176","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.34472924470178467348339361895972309628","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632084649033412400763943977604348172","series":"1.2.826.0.1.3680043.8.498.73325704415302645341730089845030935591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.51299652627602596362099115085928853341","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12642641661988102031826128876502303807","series":"1.2.826.0.1.3680043.8.498.10629298674095308335832916838368065296","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","series":"1.2.826.0.1.3680043.8.498.53408408030994292905396007688439794088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12658572494577556884626613972035441625","series":"1.2.826.0.1.3680043.8.498.16822721516022701451563241396532787399","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492","series":"1.2.826.0.1.3680043.8.498.40330199315692692937128316231210557220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12674930973831274527829261576087154644","series":"1.2.826.0.1.3680043.8.498.28943366985511345008209604832460519048","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274","series":"1.2.826.0.1.3680043.8.498.12971496560624531410696500380070985689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12690803821768205116702487219908820630","series":"1.2.826.0.1.3680043.8.498.76757732230791086468903498525374930471","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608","series":"1.2.826.0.1.3680043.8.498.81987254141381728627291257521837585224","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12700981063762577397919340901063297803","series":"1.2.826.0.1.3680043.8.498.68712823976891537890413893927920279302","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12709854414521475738435319009767427434","series":"1.2.826.0.1.3680043.8.498.60297001483317491362176157903237838435","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.12488862444740738676199064192397124510","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12718966639501121738042125605376714231","series":"1.2.826.0.1.3680043.8.498.10731785866788947862309791530005356918","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.90397673065514337951339476403517263492","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.28038508695501263241847349494597143012","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666","series":"1.2.826.0.1.3680043.8.498.54052966555455568825447169749829340533","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12759704450656741777120047828560448622","series":"1.2.826.0.1.3680043.8.498.61401496577651209468395755816911353777","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.12689825173506806036971620703093931768","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283","series":"1.2.826.0.1.3680043.8.498.94325483126964397375442255102850441720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","series":"1.2.826.0.1.3680043.8.498.11367057402078659259117556332154371521","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775769262869064592411499834441472333","series":"1.2.826.0.1.3680043.8.498.49716610474678231160428660302354280584","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.36231521173847206377664978637799986925","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455","series":"1.2.826.0.1.3680043.8.498.11926799932652337583426506061098420320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934","series":"1.2.826.0.1.3680043.8.498.11855290126799477980038028564125207988","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801308844398614687904447633432197492","series":"1.2.826.0.1.3680043.8.498.12840952989395411061636411902524593000","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.26620164973396524125636337130120857106","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12815132582382825980088023336277934996","series":"1.2.826.0.1.3680043.8.498.55603756654002996986278102850460983021","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.23764363937002879317998262593320009318","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12824438254096860470348581093672563751","series":"1.2.826.0.1.3680043.8.498.22423481667587071732768233817141384558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.85903345310002600935342530209597142696","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.40362438378729175161469311051588924463","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12864336642218457767597053831458716404","series":"1.2.826.0.1.3680043.8.498.52809422212047817160570624088078403245","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","series":"1.2.826.0.1.3680043.8.498.12880014962436922811743683992623453560","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12904190317996738774518258536997285832","series":"1.2.826.0.1.3680043.8.498.11504586581861092781000309621637542600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057","series":"1.2.826.0.1.3680043.8.498.27238687537740404489531204042110091061","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.50437321603741819993742925845555895615","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.36049951151313262992037464898897475251","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12942677240800343332600884989364651588","series":"1.2.826.0.1.3680043.8.498.52734518761580241740756826521065197095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.86925822068788896797977531849168666363","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12954163390367261226110152166469167935","series":"1.2.826.0.1.3680043.8.498.56838922864992268158939592583010631805","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12958999546496283565938189274771483491","series":"1.2.826.0.1.3680043.8.498.15840932769643900737076495681599041681","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395","series":"1.2.826.0.1.3680043.8.498.99194758479127894162891911106520151743","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12978172997053500029791786614906570285","series":"1.2.826.0.1.3680043.8.498.35072817235108436209429595800955663975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.12076638729275158891249463035024966782","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.60906759395887750914607052163363975706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12986517636774565418070546261705439006","series":"1.2.826.0.1.3680043.8.498.19950981742091618814520940477316637709","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.66709237839090860593525642852067821548","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13009271274636013067662786774946029285","series":"1.2.826.0.1.3680043.8.498.69970308795021471998546794313114469185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773","series":"1.2.826.0.1.3680043.8.498.63826919424382504029136669489188971239","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13022585036130454941964190427558363602","series":"1.2.826.0.1.3680043.8.498.98524479193308359522351034545883379357","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.10585757307102148506582411797574357185","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.14821948996953631716726505542401823565","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","series":"1.2.826.0.1.3680043.8.498.50715694066537496356614507007355830539","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141","series":"1.2.826.0.1.3680043.8.498.17694761818943864091871127651569902764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.69135403688687136280571565669705821515","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13065120731888368919957856813755701101","series":"1.2.826.0.1.3680043.8.498.72744612351627601566979955765381949467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593","series":"1.2.826.0.1.3680043.8.498.26557146639745440557183892416566444770","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13077357855769903335728068420884303993","series":"1.2.826.0.1.3680043.8.498.48746333099374436564239915696943156282","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.11646463112250602749107577257093955826","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.19460007553265267217738558325800531328","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13088550794087945662322609097092458487","series":"1.2.826.0.1.3680043.8.498.11969995319806431790246568809953466487","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","series":"1.2.826.0.1.3680043.8.498.55548847101198421664443298057077965273","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13099702881219862576749493772702888064","series":"1.2.826.0.1.3680043.8.498.11359337530563256753017645178576734250","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","series":"1.2.826.0.1.3680043.8.498.75263635986026458459192746628585878570","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13114228294359876639109410575915394712","series":"1.2.826.0.1.3680043.8.498.13031296107901096787808216982348364219","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.92311451729318889800055204134155902065","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13125910783461518266627225743585018143","series":"1.2.826.0.1.3680043.8.498.69931043307600336442241554180195145469","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13126380916005729420163981055214890048","series":"1.2.826.0.1.3680043.8.498.14790502976473759816387037568704562098","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.11778992497524987974226903180298823379","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13131420451380633781538717117200087258","series":"1.2.826.0.1.3680043.8.498.92174663229727921407801397244970826904","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13135567399913958681613600650440761032","series":"1.2.826.0.1.3680043.8.498.29800664003909507426181862873343585115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.17078849534868982992011750353128548690","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13139684164906828131777150061532685584","series":"1.2.826.0.1.3680043.8.498.45854374062190321253711107722619119619","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.73967113331348420021016609425629354137","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13150780733560495803861509587935093776","series":"1.2.826.0.1.3680043.8.498.12070474938344455445717480273428513108","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.68030285592530399462144331358290550115","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775","series":"1.2.826.0.1.3680043.8.498.79648778591143118745638331046684960511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13158644154518335426512578922874968296","series":"1.2.826.0.1.3680043.8.498.27223266024872406246712862946845613465","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13162479706670045075896993021883562680","series":"1.2.826.0.1.3680043.8.498.11777554071923272151102781925826062751","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.12341092394857002356914031491426229558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173050135651736408720687361834045815","series":"1.2.826.0.1.3680043.8.498.56608850020810698913728649702337829851","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937","series":"1.2.826.0.1.3680043.8.498.67788777061231390747507596080213718321","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","series":"1.2.826.0.1.3680043.8.498.28404140817663462611847284160690421804","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075","series":"1.2.826.0.1.3680043.8.498.29062641042433961161801538631645292348","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","series":"1.2.826.0.1.3680043.8.498.62144319944185075745913634464939146520","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13212280182729018182073794487313115605","series":"1.2.826.0.1.3680043.8.498.48055971343977755627246191020208083887","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.10620328458985189559722349458292498467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.15219473032464831095562174985863402426","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13234278816788324761159326624155958655","series":"1.2.826.0.1.3680043.8.498.92477232019285337163922097827797052612","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","series":"1.2.826.0.1.3680043.8.498.20700474899336501022485459610440320429","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.12814785020600297743415146940938245475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.61562118046114157246083271150852286123","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13263128568157735797391439005687112733","series":"1.2.826.0.1.3680043.8.498.97027152166685982651733802471903266334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13269841239774089690227054060975903636","series":"1.2.826.0.1.3680043.8.498.81078816884385454543719892685118527297","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13280017707177438711374726939683941320","series":"1.2.826.0.1.3680043.8.498.46434421769699986428788451607164497907","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.24667191054129740156202925503860325560","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13290698873574296175460151724606672982","series":"1.2.826.0.1.3680043.8.498.85362005024981664188663179251405254774","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13295231176137292542631413614945667314","series":"1.2.826.0.1.3680043.8.498.13109132308452580784591382528380628441","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13309299049346932814529962458932580753","series":"1.2.826.0.1.3680043.8.498.10220380196157596429468532710495124806","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13314276151860609710431717811242659651","series":"1.2.826.0.1.3680043.8.498.55825492374169297448181192331354103440","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13323187196249612936800308722675884644","series":"1.2.826.0.1.3680043.8.498.64397338097883625601199272668956878017","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.36973361533538448860596237858481995885","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13335786652653204791740906118815827207","series":"1.2.826.0.1.3680043.8.498.79279575156755275010986138081350367874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.17428694112690835799393392717773887983","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13357923062946813820390731946535198170","series":"1.2.826.0.1.3680043.8.498.80693156793332779257194702956819986151","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567","series":"1.2.826.0.1.3680043.8.498.14148528325732718875579978243692083577","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367691581890020433564928882602648123","series":"1.2.826.0.1.3680043.8.498.57895204327994107937752461267726725563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737","series":"1.2.826.0.1.3680043.8.498.10942826437681700365030363156314973029","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13377426362348493372112241208607685717","series":"1.2.826.0.1.3680043.8.498.88860180237439208895130394035327119754","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962","series":"1.2.826.0.1.3680043.8.498.53073862774221869923222611986268170588","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.10607504398992216268597015359325527656","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13400273795954308944359576841942016981","series":"1.2.826.0.1.3680043.8.498.11410789112243666224749206113271745180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461","series":"1.2.826.0.1.3680043.8.498.12701106680437416585401060466658446072","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13459473184684263663596121399737033933","series":"1.2.826.0.1.3680043.8.498.10038965291494057560598082657384946898","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085","series":"1.2.826.0.1.3680043.8.498.55088839686513631065885988071212298475","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13505671694940677161128783863912245059","series":"1.2.826.0.1.3680043.8.498.48922537210804069660197409162028962422","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13597463452741791628023738080583139636","series":"1.2.826.0.1.3680043.8.498.25575595483528409536832138380838781291","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.25072383915505970206154368352921394237","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13762716088334302039143651507177887969","series":"1.2.826.0.1.3680043.8.498.75232715579346798916481800567361354331","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13879163301346984697246311226647565712","series":"1.2.826.0.1.3680043.8.498.95373142064669926980550385627701697273","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.76435304813989077358090840162122032121","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14024129019390041106306062001538710978","series":"1.2.826.0.1.3680043.8.498.10963685398489849462918423926682272480","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.29842315591154130884173324180586072039","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14137431039800236510749401303688858447","series":"1.2.826.0.1.3680043.8.498.58039755239116622072398747505521908954","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.27649318594190323800086972802563368385","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.11947578595198143326999774651689900672","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14347036698014475959973071477777915575","series":"1.2.826.0.1.3680043.8.498.12354441417681753107432814809457588236","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895","series":"1.2.826.0.1.3680043.8.498.10583856058538942508518020360668069551","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","series":"1.2.826.0.1.3680043.8.498.45850695765631699443163235390571909251","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557","series":"1.2.826.0.1.3680043.8.498.29674178691985111585439769605811194972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438","series":"1.2.826.0.1.3680043.8.498.40467458821845670062190474048714921763","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143","series":"1.2.826.0.1.3680043.8.498.45513519975053017090662710556635714796","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14776241668061173271058268311868302014","series":"1.2.826.0.1.3680043.8.498.63352856598458293305538903587628746920","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.75680895973372016219791110975833203989","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14959561028578335448289993045623217861","series":"1.2.826.0.1.3680043.8.498.23523112619804934219057592655797714731","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.62504914318647034155565795033005941107","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211","series":"1.2.826.0.1.3680043.8.498.63235051741610874799192504374772293071","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15103255966832759925770334719703425755","series":"1.2.826.0.1.3680043.8.498.12751064915534281337406787378571174922","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.76743449480601223421451892130351976238","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411","series":"1.2.826.0.1.3680043.8.498.54701865570038686273513265241081936686","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.10676816043523948544126005273834168949","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.11201576449057082464891726039584516176","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.32039664756915341502006587692356990051","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.38027356970865916231145875540288522421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15595595490910196952568459534537629865","series":"1.2.826.0.1.3680043.8.498.10555392099828549633905944339837148770","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184","series":"1.2.826.0.1.3680043.8.498.12570885588677558220219963751539923740","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15851463019383290210760064201595021029","series":"1.2.826.0.1.3680043.8.498.35886100504151525419895774834438901216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16046164723877343553376490423345098256","series":"1.2.826.0.1.3680043.8.498.86074561706437498460000633575977811428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16060119389060497136231217718921482192","series":"1.2.826.0.1.3680043.8.498.11844032892710836502823802081213979899","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.54527368595735533324747291393112654080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.25505202237001650998111383858413663924","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.42424365143895266352185963707993414571","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16364873390453802195568603605191835957","series":"1.2.826.0.1.3680043.8.498.97613402841128611037029030780225742771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846","series":"1.2.826.0.1.3680043.8.498.97679739354940621879846664978135184506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.16527747624688737126118546487228060848","series":"1.2.826.0.1.3680043.8.498.13162274249699423303585473531401907820","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.12773871760832130844346942260218749063","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16589468854653713035210251187012168350","series":"1.2.826.0.1.3680043.8.498.55353410913842133598590861262929312964","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061","series":"1.2.826.0.1.3680043.8.498.29144578983940275874136607538448725359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.96815400880034569079690695996847158471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16724013341359693140331158148551346369","series":"1.2.826.0.1.3680043.8.498.63938771315697626384754495434617898681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16843381581429596234606436682284624187","series":"1.2.826.0.1.3680043.8.498.62492943397164906836122513214368401183","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.60082800552297061052313101262901229202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17049315826418217889213226790962592213","series":"1.2.826.0.1.3680043.8.498.49657157859351261251465754014883152289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17070139245390396445395414066427036835","series":"1.2.826.0.1.3680043.8.498.64093540392311076190498704490417306533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.25411625276227787762524675687579184777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210","series":"1.2.826.0.1.3680043.8.498.22965301896021372832084670550611368524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.61190112227319371309666818038180083271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.98097764372629614440286018741857077266","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17389120886685633448235711042721669823","series":"1.2.826.0.1.3680043.8.498.11929577498362228678652735431408321423","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519","series":"1.2.826.0.1.3680043.8.498.91151170359306480259769935733596721058","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17547781127981007962008163248942714330","series":"1.2.826.0.1.3680043.8.498.11777344048327463289186253547865378798","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.50611480059730944056486009629726633792","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.17691941055331862184525411493110144433","series":"1.2.826.0.1.3680043.8.498.11732967989180760655680098537090895223","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325","series":"1.2.826.0.1.3680043.8.498.51739921450454049332233713072513358332","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17836815413824722543229791044312174885","series":"1.2.826.0.1.3680043.8.498.88272503889536046439528715136485625313","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.20353631680136808681844637637600026549","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17931021655186901394796168474942485728","series":"1.2.826.0.1.3680043.8.498.37265227959941421921979555442966989419","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258","series":"1.2.826.0.1.3680043.8.498.99388169583707270193018778518471346065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18092766733837308394607532608319746043","series":"1.2.826.0.1.3680043.8.498.73331532268453216829320772194949957035","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.51776917479949404013572219975241185256","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18199465156572252531439880765733639584","series":"1.2.826.0.1.3680043.8.498.12522158029905874683173518005671831997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","series":"1.2.826.0.1.3680043.8.498.86816744093690547649210973158249456008","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18350185433331570523543207086298834067","series":"1.2.826.0.1.3680043.8.498.82455430554530281602363817416300698103","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.40078005876572289199294408070278297835","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.16655466087900999757085965363017616244","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.64305034932799882104662987032239299321","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18653838887148864162027930218548124748","series":"1.2.826.0.1.3680043.8.498.55517151751945900799081799392275330033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18680059761479598675335345641208680006","series":"1.2.826.0.1.3680043.8.498.61190862248721184768331652223336601885","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.25663978079036825555065837074369260467","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.15830344183643737530026147823587622074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.18848481788456572234982139368351538716","series":"1.2.826.0.1.3680043.8.498.22347627497092961764059556693467758173","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18931107448953327254073392032141677797","series":"1.2.826.0.1.3680043.8.498.64818543578864967222301785886581197546","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.38478500814899809365983835560984315357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.55703598257738230989484505389159878506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.39661975377917524535933309815573041599","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748","series":"1.2.826.0.1.3680043.8.498.72423158517942292200182744711054258304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19342628541833173544083802327412975225","series":"1.2.826.0.1.3680043.8.498.71122938408935491473363353873667310205","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293","series":"1.2.826.0.1.3680043.8.498.35983774880972928655011672989685511021","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506","series":"1.2.826.0.1.3680043.8.498.78008458484079048189028374670173682337","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462","series":"1.2.826.0.1.3680043.8.498.10716553757690320722316317430745837474","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.22763951614587642844868927174493593256","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.32164418214373306270279753783932960899","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824","series":"1.2.826.0.1.3680043.8.498.66628787208419603929937802751632400999","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19832308119617081676615236082965891208","series":"1.2.826.0.1.3680043.8.498.29062534460461595293915007432406619506","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.97979918279157951496734097854344700921","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.69905618915951856850020075201168200463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391","series":"1.2.826.0.1.3680043.8.498.11035287118941519122733726431905666905","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20123813962665544845986028832159432587","series":"1.2.826.0.1.3680043.8.498.96808042190949595896542232647623224479","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870","series":"1.2.826.0.1.3680043.8.498.89863571209784058452903438351005322426","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20196165276643741937915346646793735100","series":"1.2.826.0.1.3680043.8.498.93502209050112873906516217092073630527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766","series":"1.2.826.0.1.3680043.8.498.35554592625765518241292654775855428155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.64842889719588175964259843915701302619","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20427402498958253193571475926580142499","series":"1.2.826.0.1.3680043.8.498.11152730140409758884686569292075924051","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20488337421252584047930242186072775007","series":"1.2.826.0.1.3680043.8.498.26202680422718921893885592726506572254","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20521417572710861366819886462837836465","series":"1.2.826.0.1.3680043.8.498.78697720036035740968256663977621693963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640","series":"1.2.826.0.1.3680043.8.498.24562480005290220663206457316548194402","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20690227395174139480327772367312879333","series":"1.2.826.0.1.3680043.8.498.11996628930150670517107669261857912544","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20781255814255169306555078952156148784","series":"1.2.826.0.1.3680043.8.498.16837416019620271134500354070176970678","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624","series":"1.2.826.0.1.3680043.8.498.12162687547288340323398060324621118361","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","series":"1.2.826.0.1.3680043.8.498.64943353230679127273048676172764873075","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.23752082391624452374318214159798886256","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21024377993947321291280779867493407199","series":"1.2.826.0.1.3680043.8.498.99756865494595141142824145518281660651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21135201439483157418374033420156525658","series":"1.2.826.0.1.3680043.8.498.79162764317395382737197874984686007249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.24077587683640281423254266254347808266","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.20426137319785020440626365962262039833","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21403015320378798778538823075316902483","series":"1.2.826.0.1.3680043.8.498.36252964793350042124970026984052916664","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.16734111436534538539250218193695922926","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511","series":"1.2.826.0.1.3680043.8.498.15804432277982321627021249160793595240","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21605506484276750075895228202564072044","series":"1.2.826.0.1.3680043.8.498.86503892096219946924816229366996913379","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.26487734780308557989715932061314761226","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21769352980698937182060933429354012104","series":"1.2.826.0.1.3680043.8.498.22299243413996752386293441971692524927","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548","series":"1.2.826.0.1.3680043.8.498.63866269592458101917260090213134049840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21904421326144238681996713485932410596","series":"1.2.826.0.1.3680043.8.498.12319844349576654616228864926711919952","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.80447630790433074878008434467424989959","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.69243020966477058749918373065264345062","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22053727963380087706894183799325067544","series":"1.2.826.0.1.3680043.8.498.30312748347182078813153788630418377687","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.48895343607454002164603680016381270063","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997","series":"1.2.826.0.1.3680043.8.498.48315459509679717325477904281823228144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.51088542169189356832014137451995552469","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22275537949387641831010030150398063643","series":"1.2.826.0.1.3680043.8.498.10273498432404468154456110886328752651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22394208178597637441149549895512290013","series":"1.2.826.0.1.3680043.8.498.73164823067389043495794059636850558720","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.48531891317577362079185282914670007862","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22456525904575663955604077506103354060","series":"1.2.826.0.1.3680043.8.498.86677045942694059277557242033992930194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.10722922164833357390641006419664384183","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22594742481654038247097269358877852869","series":"1.2.826.0.1.3680043.8.498.35351827342679928781284462582776260340","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.31728962514254560677622471486487618192","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22678957404909201599340396039237525550","series":"1.2.826.0.1.3680043.8.498.52306955606281620616666095177360439658","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136","series":"1.2.826.0.1.3680043.8.498.72804137867779743905168944166327975618","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.12430803201351492097856885700427364667","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.84793901856599026181086541233505568686","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23063675880057833741455102729816364091","series":"1.2.826.0.1.3680043.8.498.91267745773964880080001564953586315443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416","series":"1.2.826.0.1.3680043.8.498.13576285460327562270489721560067493824","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657","series":"1.2.826.0.1.3680043.8.498.39048112704128875693285347571718089108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23378338408981699466914366379590099738","series":"1.2.826.0.1.3680043.8.498.92896822608634432460865099108134139868","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200","series":"1.2.826.0.1.3680043.8.498.10808441497077837304972121651115641182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.12872683864777938752715662978126242921","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.33329196772356826937296787326648963856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.11092699578548198220102107123595596468","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23757007145576852496599816072431075057","series":"1.2.826.0.1.3680043.8.498.58645774884044070062120436089823556525","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.63102193243767382652350096289896364260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","series":"1.2.826.0.1.3680043.8.498.38689627991957896357734981331362758500","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24012076387083114283116476459425015413","series":"1.2.826.0.1.3680043.8.498.52243868753797707438770161219910910780","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656","series":"1.2.826.0.1.3680043.8.498.76812306419496052157638090132470244369","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.42620116127406934301958947806883867711","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.10012828714165121366819225414906643234","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24302889622350917085868032747474929504","series":"1.2.826.0.1.3680043.8.498.42180356057013388711051777559705857853","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175","series":"1.2.826.0.1.3680043.8.498.99857535489029023141630947485179380194","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24502332819381293694020055836255028614","series":"1.2.826.0.1.3680043.8.498.71121256822437936986967711178028788097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24563801799946310644914912532825629118","series":"1.2.826.0.1.3680043.8.498.92711891672711597725800074199449311473","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.96575166333743397351649279401752325250","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24680736086205072406257935876858402593","series":"1.2.826.0.1.3680043.8.498.38364812503246712555475749606643273490","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.64284887074984722367630060776741723496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24866943620580666216571030343221534720","series":"1.2.826.0.1.3680043.8.498.12642305313498310472471270515490888781","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.28276525488167036824932862380860225736","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25010661650105533302140829168731585935","series":"1.2.826.0.1.3680043.8.498.87154385204476335111426394877421566801","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","series":"1.2.826.0.1.3680043.8.498.97148680767003290180310401902878522673","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072","series":"1.2.826.0.1.3680043.8.498.11184391046261216802426237951765466043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534","series":"1.2.826.0.1.3680043.8.498.11858283442034825730666548894167105665","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25252553477605568387898480443616118724","series":"1.2.826.0.1.3680043.8.498.28435334945646042857631401526218099698","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.38875784419894428861692806965902193702","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25394815405242666778213609728680620077","series":"1.2.826.0.1.3680043.8.498.14158828945766738205714675379544128181","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.47015694008705117816192116911552403228","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.25505153402978560400776672230220616863","series":"1.2.826.0.1.3680043.8.498.44921373983462834706134769006158985681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.10921948316304179187407875673704478751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25604547074218147115077644853162295379","series":"1.2.826.0.1.3680043.8.498.99274629643657997628095276398694519140","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690","series":"1.2.826.0.1.3680043.8.498.79278678489554502828992603333631226637","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25663557443141366332431300959089922567","series":"1.2.826.0.1.3680043.8.498.10390250789019313817895217781864415749","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645","series":"1.2.826.0.1.3680043.8.498.63466424699491742629609330316020998307","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25758766525934930593672922246867704103","series":"1.2.826.0.1.3680043.8.498.14327237519733606948867615466135788314","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.13059452250039708888333756475699406485","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.11142741420634722418843176875718366964","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.25915922780077732230099021320754032623","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25909078145370189195643713208718315003","series":"1.2.826.0.1.3680043.8.498.10738289822348881284081946675136600621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.86031275832282998277133984481919560914","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26004572902429209855865822977387283547","series":"1.2.826.0.1.3680043.8.498.13128465680021306032582879320554446369","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208","series":"1.2.826.0.1.3680043.8.498.67691908351427983967388616295367523122","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26150637525321126007832687915610995568","series":"1.2.826.0.1.3680043.8.498.12746956410710418596157813720434191049","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.97551944790896343748584722955899009912","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387","series":"1.2.826.0.1.3680043.8.498.79491179257444287268642192812119688596","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26408081008482310696738068933718440200","series":"1.2.826.0.1.3680043.8.498.28804963976976801431120493613433863172","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26474456265903975319674669470473661691","series":"1.2.826.0.1.3680043.8.498.93440724921378400887061085261420859876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.34406775285617319479872630245083692371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26619973716086136503589916868066348957","series":"1.2.826.0.1.3680043.8.498.10453415806397984113277370622414174033","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26725172078893286313351418435377708028","series":"1.2.826.0.1.3680043.8.498.92362309030834346387983710479629041405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.51878073793849258679080519971762533948","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512","series":"1.2.826.0.1.3680043.8.498.79141924823083869059671564778073591221","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428","series":"1.2.826.0.1.3680043.8.498.91584606422371069862227136116931020791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.62449051475390020516235069640034853403","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27025989808002468335294585261291982661","series":"1.2.826.0.1.3680043.8.498.57339164237904535737238859514823650829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.30660292325680891360698224788471129812","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","series":"1.2.826.0.1.3680043.8.498.84081674840346080875950862915446225260","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.10886797302420192744921887625254716268","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.10785799621302403794206107608958621683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","series":"1.2.826.0.1.3680043.8.498.57030730880471762578337990256473802383","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27319908980978184820438244981035567211","series":"1.2.826.0.1.3680043.8.498.90276339389404321306902775106620683722","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.88990551415673408785446326122333892956","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.52966130058305317432166084275691469532","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27460982183399364904497832933739191826","series":"1.2.826.0.1.3680043.8.498.11916518724657847459458455359632642443","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.98847459976613309943861891740708470970","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27653320597564308144389724321115208375","series":"1.2.826.0.1.3680043.8.498.66952253931204303253255656525523972866","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482","series":"1.2.826.0.1.3680043.8.498.12038520930483530618874082451506436686","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27710718386318170070605046744941324421","series":"1.2.826.0.1.3680043.8.498.60417625005518380458632332257992959408","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27723573213501259053792522608339812759","series":"1.2.826.0.1.3680043.8.498.95084503134941845884889569853189381775","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.17043192126750323455979706879758768712","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.45813213851002737858434908177590046316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27925145307825795804342879375835424449","series":"1.2.826.0.1.3680043.8.498.38872410222075350193301684118821124270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.10937081711410969920755723465745828825","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.28023795363827851445172311810918021563","series":"1.2.826.0.1.3680043.8.498.50154928259094237184643501141403378500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.81047524254423065524186088707910033109","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28134711422142706763867845558971485241","series":"1.2.826.0.1.3680043.8.498.29436645078339287703086080601909444262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567","series":"1.2.826.0.1.3680043.8.498.32644832640918820820800590488871465397","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28269350392124251309834584420564806882","series":"1.2.826.0.1.3680043.8.498.91131071940395273886103000603490483178","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.10528026688898628033252501638095191397","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.10637579596771376929524912301212076402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471","series":"1.2.826.0.1.3680043.8.498.43398628421831920450159404881059427266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.19612200141617878156234337514456602505","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28579372595938810820078670002733005121","series":"1.2.826.0.1.3680043.8.498.27461687043074380148263549706995950957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.11350268817395511587368806062369792644","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28638490997780173004029312240334483157","series":"1.2.826.0.1.3680043.8.498.89304178814031925337315300109713236329","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344","series":"1.2.826.0.1.3680043.8.498.30709596208336014818175060916719453852","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28697216332632397717077153882513537456","series":"1.2.826.0.1.3680043.8.498.64198485014916044731082964364641454968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.12157477096914155631977045611643487184","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28873180831637763124652074775631752918","series":"1.2.826.0.1.3680043.8.498.89772547104367607993513204003291374103","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.74808118701575628431852981172639127942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.30534039111322062899843601911715278483","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29055892482211392922525222851162308510","series":"1.2.826.0.1.3680043.8.498.58372833678050232516045086371005323580","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668","series":"1.2.826.0.1.3680043.8.498.25965844421813543854150667803397647302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29294899952044311440141172596030689586","series":"1.2.826.0.1.3680043.8.498.58523674359834850755287860305828291536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.11462688155640138600627693661513066080","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.12705522794762501087995976767134608589","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29418130858182890216867847991592582370","series":"1.2.826.0.1.3680043.8.498.12966533137847532533432901189975987189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","series":"1.2.826.0.1.3680043.8.498.10012633571864337465426288659061547254","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29735852958813434703969490187047665950","series":"1.2.826.0.1.3680043.8.498.12369249782618340960313829620695924030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.26088299299406851916843010050832832936","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29794301133358685635777478470865282350","series":"1.2.826.0.1.3680043.8.498.21161397102713509132001544590737564927","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883","series":"1.2.826.0.1.3680043.8.498.69311067432303848441531341870019119314","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854","series":"1.2.826.0.1.3680043.8.498.75378758569938759710085117305531996166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30104491808697888642424160917715722238","series":"1.2.826.0.1.3680043.8.498.49021516346135628319874997983043239313","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.17446352025557926339003440318835446425","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934","series":"1.2.826.0.1.3680043.8.498.98090535634793800625843536808117259858","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30233578499383083757206211752001832047","series":"1.2.826.0.1.3680043.8.498.12047646622167765557756960729817305271","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.91881937141969668937996897118887247220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30282025087399390929963075109487390046","series":"1.2.826.0.1.3680043.8.498.59659092325297263589132339602569923236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533","series":"1.2.826.0.1.3680043.8.498.90134749676300246660523146075943245972","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260","series":"1.2.826.0.1.3680043.8.498.77326222475312769862431610651413085980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30517381653283520211188050362394799510","series":"1.2.826.0.1.3680043.8.498.42976605285293134525785050148716244521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.40833257929355083046447756595275338947","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30662407196303880392459287553277761223","series":"1.2.826.0.1.3680043.8.498.17194688454077802266068129889245124026","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","series":"1.2.826.0.1.3680043.8.498.10751431427221794302137912143268483599","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.10572523697715651856876749297555543189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30800677497901063887487934675082066316","series":"1.2.826.0.1.3680043.8.498.28250182262463578766814045960361719630","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.61887330044811454949464774077880452179","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30888379005655662168586864728687777425","series":"1.2.826.0.1.3680043.8.498.79493700062862183685328047676577894171","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","series":"1.2.826.0.1.3680043.8.498.12754089206058644079639187049469108444","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.42774247744482496779779136279477609798","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31082592133742929889611857821843859701","series":"1.2.826.0.1.3680043.8.498.78243225249004452720484245056534461366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31145453746697192222724674918300432873","series":"1.2.826.0.1.3680043.8.498.40034177952254269904405984256527303405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.19524619943939382464255011286158143546","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31218039240665208717816408042644491715","series":"1.2.826.0.1.3680043.8.498.25733242799583205904269231227313576556","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31257542772176806182696506252925365119","series":"1.2.826.0.1.3680043.8.498.64583368098504208095570023763159350667","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301","series":"1.2.826.0.1.3680043.8.498.41040318930560023151777244567687625814","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31373064397225665483187291274194812486","series":"1.2.826.0.1.3680043.8.498.78309727791151614659143832636960265777","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31482725509752510642857072190147133461","series":"1.2.826.0.1.3680043.8.498.45564313869439111113290360368037245527","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431","series":"1.2.826.0.1.3680043.8.498.97699688604028357511584958574146934914","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31745644522310786426860265012373331979","series":"1.2.826.0.1.3680043.8.498.12548594251388596898054752478266769796","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31806829168516955528191705338336428721","series":"1.2.826.0.1.3680043.8.498.12341954066102220468470357350392830019","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31842732064173292828619477783097732181","series":"1.2.826.0.1.3680043.8.498.12082457636533663464556040745108670268","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.10758613201477852350245461616489983684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.11355244393581400866930616741553627496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627","series":"1.2.826.0.1.3680043.8.498.88540915864959005694954184806737958641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32179217289301203414651193280580594771","series":"1.2.826.0.1.3680043.8.498.11826685990591578685188899303032453814","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.30848831968705957297444995275247159749","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.81084171603553369816257504401663206753","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32329374180424300100441619205766536205","series":"1.2.826.0.1.3680043.8.498.27442467542624186606510507013334763019","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.12491855560053499506831196313017484367","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32501049887140121289143479773995481982","series":"1.2.826.0.1.3680043.8.498.36822350193653135956500787567492932249","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.98802492489268048026454604754427719933","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32593583441236180433075435541777683664","series":"1.2.826.0.1.3680043.8.498.50139683263246658337264433994274670388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764","series":"1.2.826.0.1.3680043.8.498.12845638814630172374669574744572252120","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32723296226443325838892542529446214677","series":"1.2.826.0.1.3680043.8.498.25144829856352106700921373016405884709","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.11646348062033205916320726976995462703","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.88561260305368825310438766403707101403","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32996271790302034744448837505729856569","series":"1.2.826.0.1.3680043.8.498.37103463075961455113324720996560122768","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.11410890577611298797120452103413710024","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.11564625333363025812007477916509888758","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.70052462564873791420956225189442078961","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.49744161340805442722138508639003380410","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33393833442961030521286257650328223086","series":"1.2.826.0.1.3680043.8.498.11922712187427140270615515885066348147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33432461096351771607862098991677662209","series":"1.2.826.0.1.3680043.8.498.70213392703361140551128833553461236231","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.60616819542595947238291371983206368840","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.33657688895130083119886800524324572941","series":"1.2.826.0.1.3680043.8.498.26755675950723714607651210943296522073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347","series":"1.2.826.0.1.3680043.8.498.54635960478016648262562703782525181225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33841907095190352493353075864604147687","series":"1.2.826.0.1.3680043.8.498.56075878695554593655776090103946115538","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.44043878307645081336201280587976993033","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33957165685646407808982253731894148028","series":"1.2.826.0.1.3680043.8.498.95798182876179338941802715006334442225","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34002185870196144582348922158474418674","series":"1.2.826.0.1.3680043.8.498.96115457493169218874851746122453049564","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.13352819128040464915458464463537668510","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.34109091435857467600499949506940613037","series":"1.2.826.0.1.3680043.8.498.47142278289081770612835043797513345520","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215","series":"1.2.826.0.1.3680043.8.498.49004818528480643159153282178864739993","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.29955195777034194926481183948750444505","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34404554752241524210538769325381023947","series":"1.2.826.0.1.3680043.8.498.16332523428726149474897913971595547635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.65601734078440043194337478015689500930","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.11787002578677014536702884278263106691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.40220839366666714879008923693690913551","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.65060460847849920403483685281776368767","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245","series":"1.2.826.0.1.3680043.8.498.12859619912105737026207613930357471959","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.13323207405466330340975863829265675923","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413","series":"1.2.826.0.1.3680043.8.498.12849596811899170993823253686209527721","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.92974752462759318583664509869145397089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.12715713876631037807026079642602269313","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.40268286263640491282726968405310939909","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35064504053970754752690367203246822139","series":"1.2.826.0.1.3680043.8.498.72661974064666938777339996418447456231","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.12293784982526005381936036891382174541","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","series":"1.2.826.0.1.3680043.8.498.26949868606421098997397284122725234057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35273590880523815297008790288567354883","series":"1.2.826.0.1.3680043.8.498.42019772063033694778110117748528938334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429","series":"1.2.826.0.1.3680043.8.498.84328909840122610990134661674770035700","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35549628666222168741574331855838621990","series":"1.2.826.0.1.3680043.8.498.64147871037492297586004718895841205881","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","series":"1.2.826.0.1.3680043.8.498.64048153978545098237732132571678934178","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190","series":"1.2.826.0.1.3680043.8.498.99368002095516669662967025280051884003","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.12389002883466851279008401873380141005","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.12574013478475882778063253394000830698","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35835033603115368194824573270370351403","series":"1.2.826.0.1.3680043.8.498.11424478652821822361860454006931965682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.87299105448159642647887775985519169421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35906584224533966870202042352292878668","series":"1.2.826.0.1.3680043.8.498.10446623880164524237696830160645685156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.13178589006851897436811439598521615496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.29030680715650317729254047387690582341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566","series":"1.2.826.0.1.3680043.8.498.49323675364332420027327876260767293750","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36013620117872095227314165026090752632","series":"1.2.826.0.1.3680043.8.498.74751193398992906298008001777566401169","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36090989845682961399757203739039806565","series":"1.2.826.0.1.3680043.8.498.20142889547960253239448605627167300779","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36158956431673597472934120620368656246","series":"1.2.826.0.1.3680043.8.498.35384541181388836783503715255340813172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36193046155628980489031133466840036976","series":"1.2.826.0.1.3680043.8.498.47847068268944803563398789978105890247","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36235892729383338522021416304853301198","series":"1.2.826.0.1.3680043.8.498.12381944849795102583366879455039382451","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339","series":"1.2.826.0.1.3680043.8.498.28431225276465576395121819461092486060","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36470828078108352859868482351276699300","series":"1.2.826.0.1.3680043.8.498.21073491679441834876351531642056546450","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.93904373709981321571318123421630173860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264","series":"1.2.826.0.1.3680043.8.498.87949045042006389928306342245703975791","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36679002788952511154610176614545192374","series":"1.2.826.0.1.3680043.8.498.47019516932276487046506489766548715402","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.65738906256354315638618798278218932797","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36825274340853796392762195174336452919","series":"1.2.826.0.1.3680043.8.498.22669877583859522490061754950669599237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617","series":"1.2.826.0.1.3680043.8.498.91084559625160040475038196427583538174","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36930256628357053046574486650916442047","series":"1.2.826.0.1.3680043.8.498.67720669879941505533150889040015618664","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37005775889857426034942587344620888398","series":"1.2.826.0.1.3680043.8.498.12131461006849384115031773794313569563","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37040910196459054543310335064167212742","series":"1.2.826.0.1.3680043.8.498.12399012553798337275965661635192029323","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268","series":"1.2.826.0.1.3680043.8.498.10177170866457836981105690302290682214","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029","series":"1.2.826.0.1.3680043.8.498.67102667009699495526973048133227506961","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37270291131952413969044156276264153316","series":"1.2.826.0.1.3680043.8.498.39101202862948576297043050262066193754","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.88685392398763004257871586836415489482","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.37392127307867238524251864751713143683","series":"1.2.826.0.1.3680043.8.498.36923468181159701148339847602169721829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.11291227501697595969276467347757330519","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37447511763482014027579634343525200849","series":"1.2.826.0.1.3680043.8.498.94431211035798469723400141633786236977","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.55133612857752858610577653955500061182","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37519350121906786941281770770693979271","series":"1.2.826.0.1.3680043.8.498.48657031384001598828512652137108630572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815","series":"1.2.826.0.1.3680043.8.498.10517537198581015558235114344029700574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37591590536870291237852764995372956522","series":"1.2.826.0.1.3680043.8.498.71169324990741130448113556989204602808","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084","series":"1.2.826.0.1.3680043.8.498.75654393164253941779259640365522467799","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37729566534337521443213426693965453255","series":"1.2.826.0.1.3680043.8.498.23941970935867844396440418134751704051","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710","series":"1.2.826.0.1.3680043.8.498.58302326172420942146499944568586107433","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37862788793597360072184753016994526122","series":"1.2.826.0.1.3680043.8.498.51219701821424424526708603929344243629","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.73583036917780805836382616029376481796","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37969790195791584440906553207150666923","series":"1.2.826.0.1.3680043.8.498.37088326434413359672683885464292947055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38013372055010421273425628837282557767","series":"1.2.826.0.1.3680043.8.498.71436268841189153542880836271206463559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.97491022174071438645280665287113499548","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38095098835657518803781210111463995296","series":"1.2.826.0.1.3680043.8.498.11948798440705762217496123680048541394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394","series":"1.2.826.0.1.3680043.8.498.12513898087317395111936740099802093930","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38238963443939714344730201261512911003","series":"1.2.826.0.1.3680043.8.498.10316308766270051334780405903821118184","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.78950294212288316034919214465025924619","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38325494489236992123219618004392575979","series":"1.2.826.0.1.3680043.8.498.68885929783222184735521596918178201593","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375","series":"1.2.826.0.1.3680043.8.498.44884033636265160235847569054397472159","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.99934153046137973925117271455351545458","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38457315909459574570206119213798163082","series":"1.2.826.0.1.3680043.8.498.12252012513903161240143461663306004558","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38518069950656326194016175319535174299","series":"1.2.826.0.1.3680043.8.498.10960624381838984599523096158190342488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.14731208371591629149586494005888074665","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.28339779434447074117517284672392474641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.11041492633897606865584039085294750101","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38701927697650476092773702704871872598","series":"1.2.826.0.1.3680043.8.498.59021773463332297251111567072270853485","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.11716184616305287029325531920730673891","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38795933423959135237935740989825843893","series":"1.2.826.0.1.3680043.8.498.71067050742571428933430484000115332729","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863","series":"1.2.826.0.1.3680043.8.498.11351225513261132991825410204766336622","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.77249539368280424121963808239405144352","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38912715505731269998129663552068223373","series":"1.2.826.0.1.3680043.8.498.14601432495901784810206395370007826385","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204","series":"1.2.826.0.1.3680043.8.498.60413341810996626390498155223199932424","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.13245658709197490671399389119846950756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39020232951771672617852805188845504161","series":"1.2.826.0.1.3680043.8.498.71260689902034255796144800214069373222","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","series":"1.2.826.0.1.3680043.8.498.11912157634582512293102503339936052975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.56929731653105849551512345028566637684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.42438160585196023644048824512994800776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388","series":"1.2.826.0.1.3680043.8.498.11814309017945133703120689351667223146","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39394842976048577715164357752399606924","series":"1.2.826.0.1.3680043.8.498.11004498966001266555592957328903938986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.10681444947500103079455959043947277778","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39547752108509751520306190293259313911","series":"1.2.826.0.1.3680043.8.498.40835314928811159365868108308215637941","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39624624539323722688093816284210287611","series":"1.2.826.0.1.3680043.8.498.10250583930723173809154682754146927715","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.54878626462839366449657754464161243219","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381","series":"1.2.826.0.1.3680043.8.498.37366106156572896886364756218196591671","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39895705984068714346274689335576595443","series":"1.2.826.0.1.3680043.8.498.46344657797566309878917619998649511168","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.52265250725198370670003971556609647118","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.11320126249681275732227507036639175677","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40086005762552976451418579375198597540","series":"1.2.826.0.1.3680043.8.498.47861105241521745954023415414194659097","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40196271329411009311273016603705560443","series":"1.2.826.0.1.3680043.8.498.56572247829918444188492919547221857773","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.93950555707061477144448715660849348868","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830","series":"1.2.826.0.1.3680043.8.498.69633098930645675095893238009264645437","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40317527233615709136328116805254969837","series":"1.2.826.0.1.3680043.8.498.88996434018101409792291168903227608329","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312","series":"1.2.826.0.1.3680043.8.498.84316905201320461285716827581864223479","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40437305053062892226978469238653825745","series":"1.2.826.0.1.3680043.8.498.98882809838033461188023061202036258714","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40487147942341753806704765355921322161","series":"1.2.826.0.1.3680043.8.498.26885458322348002108818230958511922873","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.21303084861773297057861017549695063719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267","series":"1.2.826.0.1.3680043.8.498.62172320171736907256569524369537161951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.10806261213409937116169937864290047313","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376","series":"1.2.826.0.1.3680043.8.498.10058914736132089479743551136719836886","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40765829900885943055669918091033014654","series":"1.2.826.0.1.3680043.8.498.30390299542134168637101661817338570822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.10564375790442693776625247179568584583","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40910368031948275476880475208016058754","series":"1.2.826.0.1.3680043.8.498.27567485696689005124443916287732343836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878","series":"1.2.826.0.1.3680043.8.498.11175132846165175526776981483225070294","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994","series":"1.2.826.0.1.3680043.8.498.12807109529955425779017639605948889023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.57070135958020527642815178799773392449","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41197678541534079475667407055098243437","series":"1.2.826.0.1.3680043.8.498.83906280196662902178941315615477060896","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995","series":"1.2.826.0.1.3680043.8.498.31300673181133224537485148941766042949","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41363979247025384202932112601795525615","series":"1.2.826.0.1.3680043.8.498.10603789953732967200262238533421879075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.10007290698910138166420551269680148300","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41454762422897656673710145732036115764","series":"1.2.826.0.1.3680043.8.498.87247600380245747252988811639308439227","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.57797094265453935520307311548726835230","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","series":"1.2.826.0.1.3680043.8.498.65457481837983341476740568334415593536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41640481150213029810772014344448781554","series":"1.2.826.0.1.3680043.8.498.64977252560264196576406794494400377175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.34981833200239761507167231692290221237","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.38329406043709448668212560654205243506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.41751049067043481681879071436091069601","series":"1.2.826.0.1.3680043.8.498.13005921476250742866069229623840972070","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41776233288569201013095816839442161537","series":"1.2.826.0.1.3680043.8.498.56380485730216550785108458691390410649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41852534027066121760361035269460584004","series":"1.2.826.0.1.3680043.8.498.16605351405265501318544535683311192538","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.39806870171371896357001835537089668883","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018410640565840140895012243840256810","series":"1.2.826.0.1.3680043.8.498.15739488975683791426193116366206813767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.72517258705949716884431050071119036945","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42120645248710860870761681999665860884","series":"1.2.826.0.1.3680043.8.498.12612556171031134487680149887722251772","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.64228728921682916645296793408356393735","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.11086930334636678882408024434403988126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275","series":"1.2.826.0.1.3680043.8.498.72452965543090147027534392365611614549","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42476443240785038044326483794941265936","series":"1.2.826.0.1.3680043.8.498.10938175170104981833998866331085660186","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012","series":"1.2.826.0.1.3680043.8.498.10709001997927171208142465833626398636","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42587087495729026068405824960397052997","series":"1.2.826.0.1.3680043.8.498.78749730233474915458705451034845508947","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.50663331821027604010263998316417766713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.59364910160064262446869754323551237053","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.93177164635963558590465708093429808378","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42847376403722341195083350288786052637","series":"1.2.826.0.1.3680043.8.498.72010284196786089899722256727269667059","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42896117106271721157113072562739356381","series":"1.2.826.0.1.3680043.8.498.79731873119829053161001901760751298835","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42965031665568825650795822006244115609","series":"1.2.826.0.1.3680043.8.498.11871718482431227236390319909109226997","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033","series":"1.2.826.0.1.3680043.8.498.14007011624560594781475260181014930589","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43065391907984821806263204343792008938","series":"1.2.826.0.1.3680043.8.498.11281054690325537524609260740035779280","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623","series":"1.2.826.0.1.3680043.8.498.10317954110040450028281690550255876135","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43256905750043156063416845850825787417","series":"1.2.826.0.1.3680043.8.498.12460510325058106625185559665479976844","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.26569392920702428921657672517283343098","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43355923025859051425546451466843844346","series":"1.2.826.0.1.3680043.8.498.12489642043470249957562520123750060318","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","series":"1.2.826.0.1.3680043.8.498.13000177169313725100209663912635115540","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","series":"1.2.826.0.1.3680043.8.498.90203290807717156292032600955053858432","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43587826365647713044374316837330397375","series":"1.2.826.0.1.3680043.8.498.36116704531079444428408371581101477740","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.12871308870176550749947542028708653917","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011","series":"1.2.826.0.1.3680043.8.498.98233312702516271872441130707970270444","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44002376543340544410495269060824194475","series":"1.2.826.0.1.3680043.8.498.62075178092521603695024564838463424170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","series":"1.2.826.0.1.3680043.8.498.13366143444990456279823484019829238008","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","series":"1.2.826.0.1.3680043.8.498.95803243434096582483287850726966193133","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44333089398623004498721836325723633455","series":"1.2.826.0.1.3680043.8.498.86434272198724802941548874692973780073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.13397030373252644818149333051783513670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44424196147366603683090692394096032727","series":"1.2.826.0.1.3680043.8.498.12432211029005585935535284454363352267","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.13293394390048577140275085504496478003","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040","series":"1.2.826.0.1.3680043.8.498.73903571631902768151535118724222431410","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.13105218226124748991249652507461103517","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44792591585917525576461101442761037507","series":"1.2.826.0.1.3680043.8.498.11027785073346764877330696851388214061","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44875593707781066742286566388388035791","series":"1.2.826.0.1.3680043.8.498.10813808541391234510347804590761802989","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.30115862482236512480953562817391653234","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44996883917122941421301655137158093858","series":"1.2.826.0.1.3680043.8.498.10420232357569945311925514428514953466","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032","series":"1.2.826.0.1.3680043.8.498.13787918363474090274989179847269092288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45118217412917282413100913670807141634","series":"1.2.826.0.1.3680043.8.498.59124709153936662760656187432576117252","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987","series":"1.2.826.0.1.3680043.8.498.62455488662989541359838118064767331416","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45286493874121093392171044034263851961","series":"1.2.826.0.1.3680043.8.498.11976562942124258300962735810887940946","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.62878358812644855276390191349235617177","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.91584118067125423350030115618486512531","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159","series":"1.2.826.0.1.3680043.8.498.10475356530086951355518835403524297620","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45454539422542821586939369440135728144","series":"1.2.826.0.1.3680043.8.498.52191813839532941566555392454195701266","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.28615654812133410691041753498103603927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117","series":"1.2.826.0.1.3680043.8.498.60042510425238844600220714417047713642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45644703214235636656820126625301033696","series":"1.2.826.0.1.3680043.8.498.28180902381536990165470935258108106053","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.36676111592592131504980924735239054627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.45862157701704991290669186265816463426","series":"1.2.826.0.1.3680043.8.498.12089764162143140501792850001687876044","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381","series":"1.2.826.0.1.3680043.8.498.34421871765884187281952275489821523297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46001596547902477402837300565271742663","series":"1.2.826.0.1.3680043.8.498.12193646110983323877425067772498857221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.13193887684795951337152256963472393219","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46123761840464615814604226041496278995","series":"1.2.826.0.1.3680043.8.498.67023124743321172102836018847820895876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095","series":"1.2.826.0.1.3680043.8.498.11835187370876759470516102116384460103","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.46268820642949526111807948525363811264","series":"1.2.826.0.1.3680043.8.498.76334460953828722309554712708964969867","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.52843717883742913291611627837183232466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493","series":"1.2.826.0.1.3680043.8.498.46187974150813409187153414977043700615","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46452612704507986055186132736451572677","series":"1.2.826.0.1.3680043.8.498.25690215352133586165092065907077676623","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.55841395361476140515289061478310388507","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46713978512945139851373191422321462143","series":"1.2.826.0.1.3680043.8.498.13704420167499147490970465349070099301","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","series":"1.2.826.0.1.3680043.8.498.10867223859189652465894705067035179072","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.24183429833714290129044115602716326675","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.35921281485413348014295473546172577790","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46909729013880708074310736136355969956","series":"1.2.826.0.1.3680043.8.498.11257721613073637523317832018091193380","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","series":"1.2.826.0.1.3680043.8.498.11093794355276644466175921445732001691","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","series":"1.2.826.0.1.3680043.8.498.39792331308272075174868283761978556877","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.75679703268581658988601851159205281509","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.89841607589559196716600277015317417947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335","series":"1.2.826.0.1.3680043.8.498.12324233228401350455781697513628587329","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47391919345230380216864709248400573953","series":"1.2.826.0.1.3680043.8.498.32880076675788659751798609645965980138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.92441193218151385426367401906905507264","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.11435012018203283374373062459158528205","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.59562786257963260961955599788180671759","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","series":"1.2.826.0.1.3680043.8.498.59317264803824080646039730196173813925","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.11563860492033421731372069122679984763","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47921753480592595198052407850568677187","series":"1.2.826.0.1.3680043.8.498.54686691624613803504966202537891841078","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510","series":"1.2.826.0.1.3680043.8.498.12831738864884635023097015334589736459","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48141366506505611287110229972010695331","series":"1.2.826.0.1.3680043.8.498.46271453038523367901840189288716938776","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.10225164441214828116709169975834166227","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48368929760320103375578806071106317953","series":"1.2.826.0.1.3680043.8.498.54289071194395636587717860826946455470","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48435973333068748044485658920315789799","series":"1.2.826.0.1.3680043.8.498.96211909765176921244224688695324865278","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48521453765449688946535921546652765399","series":"1.2.826.0.1.3680043.8.498.52795603072470389035738871974301283540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.32953407123364667675802576048108480520","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48750912584073496502856009873745362236","series":"1.2.826.0.1.3680043.8.498.12179608463558531690159081900438187551","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126","series":"1.2.826.0.1.3680043.8.498.95004474307830103180374707055133889388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48829151552094424673693975665028690142","series":"1.2.826.0.1.3680043.8.498.10632133550407059266223628780480925705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.88137516751686219550286653559130720190","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48946580946665031852355005294734101132","series":"1.2.826.0.1.3680043.8.498.71137653521806972309735604555464864164","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.20014359371345743551168975656863757410","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.10311827936417073401676475202687646858","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49163335688297423808344168197551387860","series":"1.2.826.0.1.3680043.8.498.12394696304676294591664815860431403657","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.11376623530894104040445888778769808302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.12088578879351395574896858299469395700","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49441835811239898312243678447796777305","series":"1.2.826.0.1.3680043.8.498.16146908487405002697987680959379478466","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.74645541540960190362832089083451754572","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49500752658775616590911569693817273205","series":"1.2.826.0.1.3680043.8.498.11765695481925969652870327858412940336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.12076461446701190788585412865504850640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.60371035535156546218600232184614764640","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.14774388084782815786774799927492695483","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.72441681528407455887554370383898306759","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49880369313850883713947031303955494125","series":"1.2.826.0.1.3680043.8.498.11628332441218388079885210593309237836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.60883810610429353621387666064620943023","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49984662916977328110175445895515541267","series":"1.2.826.0.1.3680043.8.498.68599807230663939721427271281798832075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387","series":"1.2.826.0.1.3680043.8.498.48038762034755659224895058360619980807","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50107479256735739327080544431929221135","series":"1.2.826.0.1.3680043.8.498.18736015033692177202835907569222661676","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.69142471195977839055166456389343546273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.96059154221237767954457951452869450189","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50259348432783983788186173025555513832","series":"1.2.826.0.1.3680043.8.498.97218232628071695392061446729787340364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601","series":"1.2.826.0.1.3680043.8.498.86909750854263036579964057235844657846","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.11960458314285091996875419219606000637","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020","series":"1.2.826.0.1.3680043.8.498.67629639369591919457157535706161211094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","series":"1.2.826.0.1.3680043.8.498.13124979757742095232631449789575046137","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50648934951234229265309905009982297515","series":"1.2.826.0.1.3680043.8.498.12979885159578642309592836298228340926","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.21346429899498243136109965176248645442","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.10645227211415406602523257848600065751","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.12570723016079535173711238161421810302","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.10708363758127287792717373944100536174","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.50819987120895374623148160166392914169","series":"1.2.826.0.1.3680043.8.498.10056226282936337230425086239060494094","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.12376420324927218978520514490048000901","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50882628969985977810341532747520504841","series":"1.2.826.0.1.3680043.8.498.53198834671678864036803283516457705281","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201","series":"1.2.826.0.1.3680043.8.498.68299895607862150099573324470751263527","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.84252769814832601932636910250629126664","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51143994849141407980459861759302509053","series":"1.2.826.0.1.3680043.8.498.10955424529380095842148248526690327847","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51215847567976999838748705030071176367","series":"1.2.826.0.1.3680043.8.498.46053272074776573480388347763940491521","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.64143900612340102243929121290693368106","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598","series":"1.2.826.0.1.3680043.8.498.11476977689918650013566001224334155709","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51509214978951980397275258374070843967","series":"1.2.826.0.1.3680043.8.498.10769002615066788558001754451958070342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464","series":"1.2.826.0.1.3680043.8.498.41386994982340115989440944623604069772","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51752963441899958127321113249182970131","series":"1.2.826.0.1.3680043.8.498.95461319789349931821015117560734593172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.11073369656333594529313240645394526704","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.51813125531970590883098432751115674643","series":"1.2.826.0.1.3680043.8.498.51834862027969136549908662919997825661","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","series":"1.2.826.0.1.3680043.8.498.83105772732248442835378300659628462488","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51886877282987244004912794726043055730","series":"1.2.826.0.1.3680043.8.498.98095835738263965128456272884135583888","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.78523976715093387512374411532958060698","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51985313438970133264846259111827877590","series":"1.2.826.0.1.3680043.8.498.59619053199215408145785582370200610303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.11671781416204673123536530137661379847","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52067134682915759489080529708019359378","series":"1.2.826.0.1.3680043.8.498.82706228920606688910957076947078093113","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.70263353442346164657216073643510268051","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.30428228981312562877934442002595694759","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52330656126848153803758928167944040055","series":"1.2.826.0.1.3680043.8.498.39737117933080303812462019605440157701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.12421916441633758035198376563384608967","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52437948508356178175558637950043479411","series":"1.2.826.0.1.3680043.8.498.57096356037224843152411398714471986004","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.71444950165358430782356327779203633343","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52558822670236188507569589448789513993","series":"1.2.826.0.1.3680043.8.498.10245798342150816157375713205944044130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.58065441009487041562693272605666100696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52785983814702054185130109942936184688","series":"1.2.826.0.1.3680043.8.498.13141113525414454760438307942903137683","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52846236545327527967419125989598675101","series":"1.2.826.0.1.3680043.8.498.39037128453940378161521337816047602389","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.80593937072138656251902760214146794407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53076472372145933727990308381815269368","series":"1.2.826.0.1.3680043.8.498.11461166773725605477234917500822489721","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.91668841018066234554533914267605155301","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53131430415327479511358970615140982847","series":"1.2.826.0.1.3680043.8.498.38429240521392438292783897633933138800","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127","series":"1.2.826.0.1.3680043.8.498.46116169494623710080694010413506255662","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53221818460567216467597044606040911702","series":"1.2.826.0.1.3680043.8.498.96012497362755706060050397587510039135","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.76387646705818399878024891475824953262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.11543292450071950655908719863629942307","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200","series":"1.2.826.0.1.3680043.8.498.82757101018953245916855606638952916716","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53408696789721413873026963095773050643","series":"1.2.826.0.1.3680043.8.498.11118297805637888582992470016030200846","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53528011450161116437223332448942321596","series":"1.2.826.0.1.3680043.8.498.45771618087214481407380502706149798981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53628897373416863107623485630264501760","series":"1.2.826.0.1.3680043.8.498.10651848172626159268268719522987143565","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","series":"1.2.826.0.1.3680043.8.498.29125487415979177454867641953115151477","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53797236075631231042400235117445767744","series":"1.2.826.0.1.3680043.8.498.15089327665133691289446302818579253170","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.87042875701482816251592108192128574261","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53968943793571323467397128102531786394","series":"1.2.826.0.1.3680043.8.498.31568774426265484835559853532424851832","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54049224487950229496345484554581532711","series":"1.2.826.0.1.3680043.8.498.12533546079751345593638888428197203546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.80891429104775616151359951169922066949","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54138747504377266740838999528964336002","series":"1.2.826.0.1.3680043.8.498.27095239099727296645444307520230558260","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508","series":"1.2.826.0.1.3680043.8.498.91552423618065301129429075756517830056","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54280954106925918796002359856408530043","series":"1.2.826.0.1.3680043.8.498.12477517151032482785183817213864630484","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.97487226955049925660400650150943256647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54394066317295451515247748115888976860","series":"1.2.826.0.1.3680043.8.498.73344357225258507693968917283944848599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.71821287706175455948844014993243490766","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.60741502449891981195176824288512847819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54589880111295324708950851057241927191","series":"1.2.826.0.1.3680043.8.498.13369574978407820866505646710490351961","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54662745951809363774369202018332309887","series":"1.2.826.0.1.3680043.8.498.21502512122415471154132396810696495728","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","series":"1.2.826.0.1.3680043.8.498.82209736347144800723696939688424918814","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54711709751232783991855540573189433446","series":"1.2.826.0.1.3680043.8.498.67034536979712006156021320554996610044","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54761306437713414745748829128854749469","series":"1.2.826.0.1.3680043.8.498.78955468634322882016539358406025075068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54866613282412053616997257524944654864","series":"1.2.826.0.1.3680043.8.498.89114959842603719507913617062220976584","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715","series":"1.2.826.0.1.3680043.8.498.57779725329111778328641282188764111915","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562","series":"1.2.826.0.1.3680043.8.498.52856295048809719172235083562905244238","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.71517052427829006256300643565153173913","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55202459816190607876303419067810637245","series":"1.2.826.0.1.3680043.8.498.65379016147668110627980166508913557168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886","series":"1.2.826.0.1.3680043.8.498.47534003978264496220629085804561853598","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55346033326903331672579723063380636237","series":"1.2.826.0.1.3680043.8.498.10561552202799562502304257813277869377","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.73320946181169614225914103305938333652","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55586358833903787781826406120068237870","series":"1.2.826.0.1.3680043.8.498.12904374864838118125277173033205774157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.12586560988602560621137838878991542066","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","series":"1.2.826.0.1.3680043.8.498.42874767363873759658488602820018731006","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.13231008649965465743270664141030313470","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.55856308312040330141050707531982127791","series":"1.2.826.0.1.3680043.8.498.39509301321493578881133587245514172052","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.78806100534475675743626432901414399210","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55971152627433130850649668964887611410","series":"1.2.826.0.1.3680043.8.498.26007583381951913382324640244418339569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499","series":"1.2.826.0.1.3680043.8.498.98448657856980640708091311045695832728","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56054334296747411394367010227992359191","series":"1.2.826.0.1.3680043.8.498.10402022936567342626551021181474443259","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56091473614612257999053529249190958828","series":"1.2.826.0.1.3680043.8.498.49508869598642421020617210536308261242","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.20806086221230867107185854291399671752","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56216873861595017403772627709751840588","series":"1.2.826.0.1.3680043.8.498.34508281966621336386813404126087514824","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124","series":"1.2.826.0.1.3680043.8.498.41459013450892269044517287923443169715","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851","series":"1.2.826.0.1.3680043.8.498.42484053322547867127895249957179230606","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.12763175123567033737900718908091633013","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607","series":"1.2.826.0.1.3680043.8.498.30280786001820097366330167216894627693","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56628956454784173189908113425348977923","series":"1.2.826.0.1.3680043.8.498.39707874634423311118295292945727854926","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.68446772462868340358599088563799638447","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56786556618312090924315296993756843070","series":"1.2.826.0.1.3680043.8.498.49676844442758217441794514786654499115","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999","series":"1.2.826.0.1.3680043.8.498.54207900129989939951673079558803755332","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56880070152758706555016296832842012883","series":"1.2.826.0.1.3680043.8.498.11966647046879457850894653166022821632","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.98813119090932756785459633588597669734","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.86233790345510345962049261646316525422","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56989113744241846544086409786820246754","series":"1.2.826.0.1.3680043.8.498.45280926821295529845245434579291689799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56995522547527264862064392266001620445","series":"1.2.826.0.1.3680043.8.498.73635559910128626679291530381170993429","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57074072196714325300896734167161517369","series":"1.2.826.0.1.3680043.8.498.63859238357870423900638537225378715365","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.74783104902028806564181488336643817234","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57316522861709598765962486949846744486","series":"1.2.826.0.1.3680043.8.498.94116341866881606394553776570106411945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197","series":"1.2.826.0.1.3680043.8.498.75243331746411696562018914619475916765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.57537915810250993237120953960404180449","series":"1.2.826.0.1.3680043.8.498.97888428325663166219710081540113069574","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.10847605398821380490189123082429923907","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57789603189240866041812385320399435864","series":"1.2.826.0.1.3680043.8.498.39308076454735938140295226629898000775","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57829173797358025677240411871600928097","series":"1.2.826.0.1.3680043.8.498.23852822213209693376074011054641952289","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.70238581068038484606004875106645765469","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.35125771912190484743223077468901663121","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.19246634040253279997620827049388393497","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163","series":"1.2.826.0.1.3680043.8.498.60959968543814269243087420495025474468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58261214953729037661916565473101636690","series":"1.2.826.0.1.3680043.8.498.15779641767441969448009726222009560563","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58280349821647977336008109443058641195","series":"1.2.826.0.1.3680043.8.498.12233699475559550234659210766367010166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58366682083795592678613986311570432419","series":"1.2.826.0.1.3680043.8.498.76454807920106100786737958197339230017","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.37358282697411046165730905858678434812","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482152742263188503728990365891896621","series":"1.2.826.0.1.3680043.8.498.94588191766096782951328921394018982349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.95457702076226811068121221562758814441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58604600600114899516493668210902251434","series":"1.2.826.0.1.3680043.8.498.30832971802893214065005563825537473306","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","series":"1.2.826.0.1.3680043.8.498.87937682542027544271380430664875340154","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58710813820037448352310278234947865485","series":"1.2.826.0.1.3680043.8.498.91057057444556646805479501041818648582","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.57255347709103488498123732458138828766","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.91574050285902522689150219451871736996","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.51872287696580334666633304531367839780","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.10032031932490473788325421579189340980","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58965370475364247222892582428665460530","series":"1.2.826.0.1.3680043.8.498.37888308032129682077291982644115157349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59047534248122071167153110802170658177","series":"1.2.826.0.1.3680043.8.498.57956649314340183532737577282032222705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","series":"1.2.826.0.1.3680043.8.498.95479336111698330195838643959521879607","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.11075607919693695220736985549932816712","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59309239681970150083311665704264518844","series":"1.2.826.0.1.3680043.8.498.62216716655496311194584067055159855724","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348","series":"1.2.826.0.1.3680043.8.498.73768037868724215735301652556469876948","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59475691595504590121311599301170361869","series":"1.2.826.0.1.3680043.8.498.69175689528989799640157202128746512982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369","series":"1.2.826.0.1.3680043.8.498.45021010001973894025800598478983021748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59533384841491000088839469462294817273","series":"1.2.826.0.1.3680043.8.498.21807466302426369408412242062422132517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","series":"1.2.826.0.1.3680043.8.498.81644541276426391996325075918090524917","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59826277112503440743678968423276737254","series":"1.2.826.0.1.3680043.8.498.11750945083725657768194881469654632286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59864697179957009967756900038846073439","series":"1.2.826.0.1.3680043.8.498.16375819545286759114036412351839237511","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59932425615255727459402684084295081504","series":"1.2.826.0.1.3680043.8.498.19064072488710228344628373840389923804","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378","series":"1.2.826.0.1.3680043.8.498.62509360084332038760881116550120375248","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641","series":"1.2.826.0.1.3680043.8.498.40384564188193987944922192288813532094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60088605286413557861235417321985538357","series":"1.2.826.0.1.3680043.8.498.10380307356496113359516497045557033536","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.12546866856325234823080984753588342895","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.39376041165374446601324127572528791268","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567","series":"1.2.826.0.1.3680043.8.498.36644615406456256181222635833360656622","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.24197731446247489481403575431043326527","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.71300708969982807461345818749805582107","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.51975478343830714021292488048706235470","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60492772672040498730937577391401159109","series":"1.2.826.0.1.3680043.8.498.15745254194521430617835664195446498999","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.83544785178489384248256067120299180954","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60633008040453481523409852849000823589","series":"1.2.826.0.1.3680043.8.498.84632857199445921204655967513177264332","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.39093982281270077365561737739065232359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60924385750386872880015870524131415592","series":"1.2.826.0.1.3680043.8.498.56508425552996266652695238038893728835","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223","series":"1.2.826.0.1.3680043.8.498.22393720776362647619538991136628150644","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.93194990917857204179164682948104573753","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61148578767041261085865406067025346106","series":"1.2.826.0.1.3680043.8.498.50588298943671099353408694257073846472","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.80574838779868748489557860895113480549","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.74330815573707449016870926994813224297","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.31655217979000264258284655333173698015","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.12192381842557175632442104282363137596","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933","series":"1.2.826.0.1.3680043.8.498.25997572279775847284201495320127620720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.10048002059835617830123248725640685278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61630894340791204767579628554826500984","series":"1.2.826.0.1.3680043.8.498.39850746581403507277519682053740317737","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812","series":"1.2.826.0.1.3680043.8.498.36035710136466064224886103383894182045","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017","series":"1.2.826.0.1.3680043.8.498.53467903191220864639202955667668267598","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61981891095235024266625598999854298547","series":"1.2.826.0.1.3680043.8.498.94599308704376951718574275831303523546","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.39314494064031642789612465101728623803","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.46666737775301654692257137363353206906","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62217447760619326492634156203608889396","series":"1.2.826.0.1.3680043.8.498.10415750892381709193896824650854791466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523","series":"1.2.826.0.1.3680043.8.498.11216532200589628924438852110798768667","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097","series":"1.2.826.0.1.3680043.8.498.10337498269211232457136986979853287425","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62406588098851994968957074186291720431","series":"1.2.826.0.1.3680043.8.498.92078651978290547123228777096120183503","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62437671010172828772205139640126159272","series":"1.2.826.0.1.3680043.8.498.81174073741195925874373291151789643073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758","series":"1.2.826.0.1.3680043.8.498.91126493424902197532068387825034219032","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.37520573812479182452824358312509875659","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62782163204193388751869649423542931657","series":"1.2.826.0.1.3680043.8.498.31499216330593626273856867560089967390","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.31771368988040042069068831971196296890","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.62948920639798990757395955917890550512","series":"1.2.826.0.1.3680043.8.498.93176996635026090458318208331253888750","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.34360659692856146755153460097635945939","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63069333313670617776227577192591993067","series":"1.2.826.0.1.3680043.8.498.84024556702549443647949105849103928530","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992","series":"1.2.826.0.1.3680043.8.498.62951922811318377360525753451250565855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63195088465290711580668024587233364997","series":"1.2.826.0.1.3680043.8.498.78896819574006278454550057131704964423","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952","series":"1.2.826.0.1.3680043.8.498.46310977263074732309232469035757264761","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.23533450479904603295103108892705845321","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63279205896049798083339413931561428540","series":"1.2.826.0.1.3680043.8.498.90097682845842928283332514828725011507","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179","series":"1.2.826.0.1.3680043.8.498.12648352718906401017941962566617442724","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.12290675834286702825951718014883405220","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.10288622525641624692432373179101443000","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63521033783841675104013677430088400343","series":"1.2.826.0.1.3680043.8.498.69595706427210978730878222923825889229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.11660354535748994825502857648951332362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.43208804530426359142659570489091211430","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.63584266247144369196126771996126173497","series":"1.2.826.0.1.3680043.8.498.77802863167492521733914559477254331229","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.11647492351994738026192781180536503463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63849051301265505051181245915656194652","series":"1.2.826.0.1.3680043.8.498.38639091816096582739777967208449817760","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.94159108194169999083165058615767925598","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.92275457609564505719840286003323646902","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.61440899752885841992342211519564310574","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453","series":"1.2.826.0.1.3680043.8.498.76464933211712135417193862157471764296","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734","series":"1.2.826.0.1.3680043.8.498.52207762310760185259067815806896479241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970","series":"1.2.826.0.1.3680043.8.498.35961736646812769808989676951622781389","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64397747170052970974000650185716672477","series":"1.2.826.0.1.3680043.8.498.21004704562019126645213177822892991236","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384","series":"1.2.826.0.1.3680043.8.498.11816082355993551720792772938850285775","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996","series":"1.2.826.0.1.3680043.8.498.53584340005450851968411455330833215353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64527059976573114931020581328924993069","series":"1.2.826.0.1.3680043.8.498.66384061581742012909314306481332185213","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.74520305644373731111005396813044256512","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567","series":"1.2.826.0.1.3680043.8.498.27341011620636058379210755393707331914","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64752580249608101443758975861213065048","series":"1.2.826.0.1.3680043.8.498.85425071758173895031556449522651678168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64917073507426524387269500871970032783","series":"1.2.826.0.1.3680043.8.498.70305371338798062743136366480032946679","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461","series":"1.2.826.0.1.3680043.8.498.97539321718728376896555815639768420288","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65085954283273471512936799274064949557","series":"1.2.826.0.1.3680043.8.498.30013272915674461759932000653521543568","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.38244874567297259966411292233609075448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.11851035075301709990846507707824990897","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380","series":"1.2.826.0.1.3680043.8.498.12352517629725689278593639457920828386","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","series":"1.2.826.0.1.3680043.8.498.80152713658455526555383054835799927275","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.10729468467248128088219365271137911766","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442","series":"1.2.826.0.1.3680043.8.498.42105261311130763028328836648464742077","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.56758532595898003402522095210717256652","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65667678378249093118816281743956646856","series":"1.2.826.0.1.3680043.8.498.81273138270784050750778304219810099974","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.79310517219360681501640667011917023076","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65763261177572072944828171262268039852","series":"1.2.826.0.1.3680043.8.498.87341450927401221245596755001127297607","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900","series":"1.2.826.0.1.3680043.8.498.26225579987805845117102259031564456094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65834774574423727070722226151408235786","series":"1.2.826.0.1.3680043.8.498.11055767268994596483900171149654699916","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.11230972256454137931893755559637362763","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65959182350599581687336889990076028593","series":"1.2.826.0.1.3680043.8.498.97283367403163523723920448288576720123","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.80564697018960444549309349747475631126","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66065279867975557954375204776779208827","series":"1.2.826.0.1.3680043.8.498.32888058490090603929684691773239827379","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.22725777903896729880944666205750042625","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769","series":"1.2.826.0.1.3680043.8.498.12186284138382232794801177790233787165","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","series":"1.2.826.0.1.3680043.8.498.11946547048444058072632025266992480995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.83991480776528295056503130875912005085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66334686028272068369345540785320291695","series":"1.2.826.0.1.3680043.8.498.68897881993506986588905366885090064497","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.11283872586810260827563244998404744391","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66510565421813721258189101326477318907","series":"1.2.826.0.1.3680043.8.498.47859841658098562420502827127295464176","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.31068480527943845748655516886874313816","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66737859593831436259089736126768464771","series":"1.2.826.0.1.3680043.8.498.56959561152557854973528935905694977045","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.53824311448655036388935642004167452399","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.66973089869591928349547584554030589647","series":"1.2.826.0.1.3680043.8.498.33105188897240176313001312363477842155","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.54396157650197156521172294293317780349","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.83514004643308074711007986871790140824","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67189969532330535369638450306601659967","series":"1.2.826.0.1.3680043.8.498.30984200874962348797517899619151379673","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67281816624401881515444587844116136597","series":"1.2.826.0.1.3680043.8.498.15256249297671993308507552937996793505","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.11256507502319064480290461775278537357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67404282038417089622852480023919627784","series":"1.2.826.0.1.3680043.8.498.28688257763144870121484968046787572114","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332","series":"1.2.826.0.1.3680043.8.498.92090556159111707459534990794537226826","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67527486841399075490744039058207332725","series":"1.2.826.0.1.3680043.8.498.12938512934721126022073125951598429829","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.11296225081190058765104158236913370415","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.11980766028743579591708396775617488914","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67705068860320494485379949506807098172","series":"1.2.826.0.1.3680043.8.498.20532746716529135150810824071564289917","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.12273027129851884294501557490174659723","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67775301856843451215029610725114809378","series":"1.2.826.0.1.3680043.8.498.78355323882069722074603465638357956674","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.30440044131637520674266510116206655755","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.24395299166766623570121061580549148247","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796","series":"1.2.826.0.1.3680043.8.498.12182789844368380396266701609914897915","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67981157392064878495682078197161506255","series":"1.2.826.0.1.3680043.8.498.12875804464444715918718676777232738119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.27734438161585311032803371130892501460","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.98000712634920106541116999323121007423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68131564130881160284492534203572971623","series":"1.2.826.0.1.3680043.8.498.10070752204992084677370638694884209737","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","series":"1.2.826.0.1.3680043.8.498.84752741560208733741207919598182290819","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68176786698913278568384612123261229708","series":"1.2.826.0.1.3680043.8.498.42736763587251753312575036556156370443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.11562962170537674765715745568377123720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.43623687623502374602127114461392455236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019","series":"1.2.826.0.1.3680043.8.498.10729512386339014477403681137018139605","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68434323590918408241182252996136001891","series":"1.2.826.0.1.3680043.8.498.13010049157711732519071486398572045185","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","series":"1.2.826.0.1.3680043.8.498.84940139125759624795264602254896434093","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68524920268469128133836299591793117756","series":"1.2.826.0.1.3680043.8.498.15653991228270958941139808085483551467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68576465611266620513305434082656807028","series":"1.2.826.0.1.3680043.8.498.14592434379858044502275797908573855488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.61253494943679539913840879535993196878","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.13233734589945197428950604933246631221","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68769474672202195228430674317444754835","series":"1.2.826.0.1.3680043.8.498.92014931816257938935651731086662408144","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.12411348686003715083417781390772840354","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899","series":"1.2.826.0.1.3680043.8.498.34294482567575570200768695013215080530","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68969263295435881669720950631709959471","series":"1.2.826.0.1.3680043.8.498.44362139256399187771112010137323059864","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812","series":"1.2.826.0.1.3680043.8.498.12586988032176643636935724938974131309","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69131433601281694165384475765166382765","series":"1.2.826.0.1.3680043.8.498.74319258473430289826703848450377515923","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087","series":"1.2.826.0.1.3680043.8.498.82769038267931876368996510217949569459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.10726881629427324222207954820010850001","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69392348385274125385290015404144639002","series":"1.2.826.0.1.3680043.8.498.25777248294420762596105146956596348317","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376","series":"1.2.826.0.1.3680043.8.498.42467720518292507590573966195804584854","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69470130842235635628917024269527211321","series":"1.2.826.0.1.3680043.8.498.27136064574534708333735313359225056132","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69544085715595825083308172629360899339","series":"1.2.826.0.1.3680043.8.498.65705205611632208835103935559443461863","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.22763872061269704141709213169006750732","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69665129458102359604411189244405850661","series":"1.2.826.0.1.3680043.8.498.28403877344399784966976582495046909483","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69729903409900053570304330127687235813","series":"1.2.826.0.1.3680043.8.498.11573770421625950588394553391484731043","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.27095533014852237397037677143458138573","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69917156795941398965793962392683899162","series":"1.2.826.0.1.3680043.8.498.84190147828757020757192912927120766087","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.12494846540080284653897362124520350683","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363","series":"1.2.826.0.1.3680043.8.498.12321647665107714687074100668865328324","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70271042033022764422915852922035168712","series":"1.2.826.0.1.3680043.8.498.11821718848531664461302333922404623156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.28502197910923189379217871513048732477","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","series":"1.2.826.0.1.3680043.8.498.66621956789621485180080108826827057412","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70435722790459313233345196619727661231","series":"1.2.826.0.1.3680043.8.498.16608119398116369513514521574289680536","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.15811496096376163244593675490601862815","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70531311408047627903498280968772932092","series":"1.2.826.0.1.3680043.8.498.98414595323975784127069228468147724877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.64794562198820926519976649228834975370","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70705483212961922828901596159131697717","series":"1.2.826.0.1.3680043.8.498.56090386534874211039822859646254977783","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70831735578941318362881379802693293952","series":"1.2.826.0.1.3680043.8.498.52762354838848410441802175831955057536","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169","series":"1.2.826.0.1.3680043.8.498.74932835192518523653266702023631051455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266","series":"1.2.826.0.1.3680043.8.498.16491369239368416334006666490725133192","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.69848368486139784071493403233139052486","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.60964603000189189106826710100466592080","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70966951020892708303866968206211604316","series":"1.2.826.0.1.3680043.8.498.12326496107887723587954327737654619408","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199","series":"1.2.826.0.1.3680043.8.498.99174828991951154586069345961389569094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565","series":"1.2.826.0.1.3680043.8.498.21336179097293461222939340206162486173","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71262022207903748073532915013888861999","series":"1.2.826.0.1.3680043.8.498.73198426841800128878232049460564638109","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.10529624489456491594770648234546096051","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.67220440302128360381009998580308207382","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.38773861782109034698315275468060727463","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71671013221669628598634284971059228267","series":"1.2.826.0.1.3680043.8.498.63296971792528229050450397384140253876","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.11979871641763410370600780440617637861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71709477515434076105531625950744135656","series":"1.2.826.0.1.3680043.8.498.12489409568047857272638263089032890492","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","series":"1.2.826.0.1.3680043.8.498.10118319101055882198804872896810149125","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71766161038805688022754893134126528749","series":"1.2.826.0.1.3680043.8.498.67517822333122963955596108044931189166","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.23264846995701567467288938666324098421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.19499151113559485426805029708289659899","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017","series":"1.2.826.0.1.3680043.8.498.44229886713623082812345525267554549129","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72008869367139958325145888733651022033","series":"1.2.826.0.1.3680043.8.498.14892620533626703566705784424481723074","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.90049148824939552953876477872512837690","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.41987659697326880209452278140176892549","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","series":"1.2.826.0.1.3680043.8.498.53699091485887018459595768657038067925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72237065473467452226073540784070938342","series":"1.2.826.0.1.3680043.8.498.71484940280680118581870767636112327156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.57636828351303793369267097342555623084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72276631472244076773626607623783737687","series":"1.2.826.0.1.3680043.8.498.10289301696635697964274203744235560641","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.34665788267313510182714468906340921374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971","series":"1.2.826.0.1.3680043.8.498.56556448109758293732575941668488064735","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72494404099796908482534991590660770670","series":"1.2.826.0.1.3680043.8.498.42848460349023699564490343025141674968","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.34296781286778338547502085931927110278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.10975979156342844176072029132403136773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72680485820462987585370556994612687540","series":"1.2.826.0.1.3680043.8.498.18150660771917756191498569315685995586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.11521291817852373760747081888230966355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72786773761211751423503785260617678955","series":"1.2.826.0.1.3680043.8.498.19146582493882245170899189353387065874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.12116336645372311899794252097467067713","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72896348955280307080916504120667484813","series":"1.2.826.0.1.3680043.8.498.51647520396459203529031098896582055995","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.10755527372032080098563843339295116249","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72977302282583186913749000016284027331","series":"1.2.826.0.1.3680043.8.498.14723997014712663988611463218652671509","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.12058419086716871639108469478818986262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.38045873288836588191909402251884936144","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73101997284755549124715401557235981642","series":"1.2.826.0.1.3680043.8.498.13134458982249989620346237046615986432","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458","series":"1.2.826.0.1.3680043.8.498.12047580885277177081236197611110463074","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894","series":"1.2.826.0.1.3680043.8.498.35872496882990268188839732034985087446","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73242978867957397254276973409271434573","series":"1.2.826.0.1.3680043.8.498.50967391847810036927106561471325673057","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618","series":"1.2.826.0.1.3680043.8.498.83408097786893956965791285910821371928","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73325959289998105909842627876607163467","series":"1.2.826.0.1.3680043.8.498.27917336357729364545398738589907193303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.59911946024688876550852643052027029106","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73478948866451045346677275674140339803","series":"1.2.826.0.1.3680043.8.498.50462935157298747214558265217646107712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465","series":"1.2.826.0.1.3680043.8.498.10193661124752628478288016512357277937","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853","series":"1.2.826.0.1.3680043.8.498.10905078184476939692734957020544013693","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343","series":"1.2.826.0.1.3680043.8.498.15142024856207030416910560774290641589","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73716752081143767997507440278231972668","series":"1.2.826.0.1.3680043.8.498.79991086654008999210672467829406036703","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.78158700131647436580534539152426513644","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73891117356545433634964184054304516063","series":"1.2.826.0.1.3680043.8.498.29090231385915229379470835757865030097","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.35787727436618809658132087696985070351","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74078340346189210015785395961373387782","series":"1.2.826.0.1.3680043.8.498.61624938591049765756022445015923422337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74127742661271965093205833471645893520","series":"1.2.826.0.1.3680043.8.498.83031198839614711225854399683062733756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74177736542522869266952610634091628727","series":"1.2.826.0.1.3680043.8.498.52929831732799086188685642516927980046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.11642699239358421305672942692848781255","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74222294235046321147381553078000820604","series":"1.2.826.0.1.3680043.8.498.11318774961440109803099731432922277170","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.12948979104800938394948384787879049024","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.73204999765566785248033524498053377492","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502","series":"1.2.826.0.1.3680043.8.498.12203377230986336379402891285266763511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962","series":"1.2.826.0.1.3680043.8.498.13354881933648965517004304255737578044","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.91688705279440083279377541765215009278","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74622912009754642425258484521795653023","series":"1.2.826.0.1.3680043.8.498.11926060532881759233901588214389297689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121","series":"1.2.826.0.1.3680043.8.498.14849498097086320937035963082817281194","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74855168460956358728708865599401983005","series":"1.2.826.0.1.3680043.8.498.73333307685662975573258006059674001127","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74872419726042767543822107291861307866","series":"1.2.826.0.1.3680043.8.498.10483248699494899396446913794271889551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.36107728595669054102743354470416367516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697","series":"1.2.826.0.1.3680043.8.498.80962392198664941740691741574091642408","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75186274888329123179129240384960518861","series":"1.2.826.0.1.3680043.8.498.61822035479910929541857002475079521977","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.14150919612147083715320168652349451393","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.13307729860672197867066293071167936266","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75418637314688036152637525301534342254","series":"1.2.826.0.1.3680043.8.498.16859416473048475187877507619333045350","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880","series":"1.2.826.0.1.3680043.8.498.14675738556104395408591249215218989150","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75522002047190283617659083787085821064","series":"1.2.826.0.1.3680043.8.498.50547451026360915005009340342963294586","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.93380513379297637770982274418857310425","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75605038901688937188489854019478082759","series":"1.2.826.0.1.3680043.8.498.10134254916109720422007293982954307609","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75639423607035582240174986752280730347","series":"1.2.826.0.1.3680043.8.498.11835261916733248605595822879405448345","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446","series":"1.2.826.0.1.3680043.8.498.56655403951332427815036942150978393568","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75746520601616488682662195284941583868","series":"1.2.826.0.1.3680043.8.498.11875357488150611640589994641026847480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.51617475805298849002445007648174109604","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75893915698353338311947268382839152877","series":"1.2.826.0.1.3680043.8.498.10369649134908761648651898882497266276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75976388060969166980873144500539735332","series":"1.2.826.0.1.3680043.8.498.48756803122147446236748447542810975719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.31198796654542963477216132521393078613","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523","series":"1.2.826.0.1.3680043.8.498.11243566688247944462266385623246911957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606","series":"1.2.826.0.1.3680043.8.498.17187837310750226398246372901726939039","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76190704217737835083438638928129775625","series":"1.2.826.0.1.3680043.8.498.92186576388252464642730710005291675218","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.11316051418338834504545273651337232152","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150","series":"1.2.826.0.1.3680043.8.498.63603179758847605598703076569032096687","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.76525107909962775502207829870873494024","series":"1.2.826.0.1.3680043.8.498.79857394204870519968316101574025525079","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76545468257774231281618306952011717992","series":"1.2.826.0.1.3680043.8.498.27599432228175709284426659229340865705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.77664898384390515671155002271417025654","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76742750470853901575934097885382242952","series":"1.2.826.0.1.3680043.8.498.64206509253670872568366460891964871107","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.12175740666985223056107607671355876372","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.76903562636417038907113894538863784737","series":"1.2.826.0.1.3680043.8.498.90029678755590747568910788840665176415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.11904445421903998815264575437198057958","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77018355356797290743788712046815540223","series":"1.2.826.0.1.3680043.8.498.84358452639216158557419821166596077639","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77058587552329832558354544874120592826","series":"1.2.826.0.1.3680043.8.498.48724426424617453265001083265492992809","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","series":"1.2.826.0.1.3680043.8.498.50978776797771002609688876524703572583","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77164129615096222757740198026718214733","series":"1.2.826.0.1.3680043.8.498.58851382109072238942826810942723122805","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219","series":"1.2.826.0.1.3680043.8.498.52857324811275212811161669879801938836","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.49713503810677684813208613499205273295","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77283634197289382453511269950488893300","series":"1.2.826.0.1.3680043.8.498.20236709980801107866835573132627894999","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.98284164652893664638471296991030435965","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.78258700914724339615095082290322844598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77378803863827312829317008636359414012","series":"1.2.826.0.1.3680043.8.498.91644878751298032382231294243084436743","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","series":"1.2.826.0.1.3680043.8.498.57132734559707891289618176235589008207","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.10113709949039650021092206556195650989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77508762938574470575737016745357126587","series":"1.2.826.0.1.3680043.8.498.31301059990810815434963315138879422900","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","series":"1.2.826.0.1.3680043.8.498.37934535658417600459040554857208988166","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.34711195917739588444595321905995402101","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059","series":"1.2.826.0.1.3680043.8.498.56508241332212087632150397023964602109","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77628479664122209492622263818114972176","series":"1.2.826.0.1.3680043.8.498.47009954941506781250368719120723145138","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.69287400625374196048583992376469618497","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.46116325391898965061777362492549676571","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.53694457980434257041384412165458503873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77843765997593346714211135369376676497","series":"1.2.826.0.1.3680043.8.498.10029180059030247801090489398987423754","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.13270717060869032062830698632732962333","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247","series":"1.2.826.0.1.3680043.8.498.15847369523847923014771568387663172488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78072765790191594685273447833272737429","series":"1.2.826.0.1.3680043.8.498.31382515594248335495451580611829725569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78084594560404633421210837175859618437","series":"1.2.826.0.1.3680043.8.498.12539477742557379336731626349795112670","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.10085396250902427376096097790529352199","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053","series":"1.2.826.0.1.3680043.8.498.15673155071743431612945948946125588355","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78286395605671977872377352305890466464","series":"1.2.826.0.1.3680043.8.498.11075482269567949878811069137829939607","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960","series":"1.2.826.0.1.3680043.8.498.79192704852056610151817974095132221167","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78476873086705419136436170130940915568","series":"1.2.826.0.1.3680043.8.498.12457759281756076796049545657329436518","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960","series":"1.2.826.0.1.3680043.8.498.23118100718995795697309553268275719544","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78657055734742806687703707756153505756","series":"1.2.826.0.1.3680043.8.498.10083122843267350812887063988960514349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.55080419060164220366853054525097217769","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.11209805925237684981503342044595343744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856","series":"1.2.826.0.1.3680043.8.498.14772727715597036988326225986832532794","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78968420394708302953391711477016139949","series":"1.2.826.0.1.3680043.8.498.42836015480153146348721423685788276855","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.61457190924665529503047823518495376009","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79080521163097116147379189510104170611","series":"1.2.826.0.1.3680043.8.498.50349517242667053869620794254932671651","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.37018232079800969348138543358437222396","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475","series":"1.2.826.0.1.3680043.8.498.12747454602771318439978416617854709176","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79245771997885598399203314109623928718","series":"1.2.826.0.1.3680043.8.498.44259227940602614355175006076886399773","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79258497507223824227830886693749460348","series":"1.2.826.0.1.3680043.8.498.31513439208721624768338510064569761243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.10945096319356640858111424094286639289","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79326627674334637493320764205397279674","series":"1.2.826.0.1.3680043.8.498.11871165224051826646523980790993465822","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","series":"1.2.826.0.1.3680043.8.498.65398420109510153490823951263267155046","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79560170727391465844958390474997798803","series":"1.2.826.0.1.3680043.8.498.36816044530187439559184091528261146422","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.22563981180567210481524526103875276850","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.79648517065511897282938523065537768426","series":"1.2.826.0.1.3680043.8.498.14157750861634906113512471404975887287","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","series":"1.2.826.0.1.3680043.8.498.10323924710937936084351423723823806612","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79829033430619597320769579608450393290","series":"1.2.826.0.1.3680043.8.498.11976741738234981705222413406140234569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.11647227498179654880675350263898238481","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80146250663015309456767479960653638934","series":"1.2.826.0.1.3680043.8.498.11389753140320030640841487532486233046","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845","series":"1.2.826.0.1.3680043.8.498.35208615812602145822588263403024650944","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.10016070672456816867522051160523666231","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299","series":"1.2.826.0.1.3680043.8.498.69222285455172111415698021996749514072","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80312559977515578735238310472949287111","series":"1.2.826.0.1.3680043.8.498.97238531061501162877683952880525446450","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.56617256137443389204982419481433311492","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80433141990021472112645841723991584182","series":"1.2.826.0.1.3680043.8.498.71849341004542235180016355418333716051","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80492324225032367918842505447056622561","series":"1.2.826.0.1.3680043.8.498.41981435082793748982017432010696610394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80557835452734362083552169469427388003","series":"1.2.826.0.1.3680043.8.498.10123028195479555309802959914688757346","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365","series":"1.2.826.0.1.3680043.8.498.65551371659767147132628416887947080213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750","series":"1.2.826.0.1.3680043.8.498.13005890291656039529157052901150891232","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80651977424179089141346174053751183506","series":"1.2.826.0.1.3680043.8.498.29057132806344319307792324125947222187","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80698510014594772882040509716764819716","series":"1.2.826.0.1.3680043.8.498.58444667221543571537343531056404323044","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.51000590708100505863445475580362379424","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80851365368114686105558347899990395309","series":"1.2.826.0.1.3680043.8.498.11978168513445758338850076607701656221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","series":"1.2.826.0.1.3680043.8.498.50024863835631829426308162077234301548","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.29707052960329851604338563290650414145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549","series":"1.2.826.0.1.3680043.8.498.45495597569888531691635389169138913323","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81096597169632636454986988944050298701","series":"1.2.826.0.1.3680043.8.498.62317886753110796594461074104009340193","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.34889641833547478523483338460408770802","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.26749957507791318502604646487919110076","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.94263995612690845107507613870412964770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81351574724227319632426427855692606694","series":"1.2.826.0.1.3680043.8.498.13139870725795316444891987863861172221","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.44299244907703907999594163179355474119","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81475058647553697318837239968139077272","series":"1.2.826.0.1.3680043.8.498.35141551134973678980109740417044236693","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81567030584250750745410094870790052181","series":"1.2.826.0.1.3680043.8.498.10170191940901303824605512849639499467","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81629704995225424568187579791704967233","series":"1.2.826.0.1.3680043.8.498.92632529575479561301185992863540584333","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.82069403207277833908644996280082537245","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674","series":"1.2.826.0.1.3680043.8.498.77946984430148702196997132936417542483","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81740348974208459068937862799489495782","series":"1.2.826.0.1.3680043.8.498.64987341565764184499333932580230291930","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.72872027878747496661809027185898300873","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661","series":"1.2.826.0.1.3680043.8.498.25727005186661317100454867697688652637","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167","series":"1.2.826.0.1.3680043.8.498.86209256339461058816274861429112384816","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81957447654112577010261210572986132211","series":"1.2.826.0.1.3680043.8.498.13846155260108346840943074548917483128","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.35631073382777495649878539400180273362","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82012532783011843175172575198326783678","series":"1.2.826.0.1.3680043.8.498.45873455409596223197041469624301442283","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","series":"1.2.826.0.1.3680043.8.498.21428810408571112428733981355929560157","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","series":"1.2.826.0.1.3680043.8.498.48225142523716606731325544800630788645","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82169173362972793731001247413129649263","series":"1.2.826.0.1.3680043.8.498.12207353112594306203854158037956331830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.16703940183684735681300561727535315925","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82447100001417065190678617013741085481","series":"1.2.826.0.1.3680043.8.498.21363990322152610951063263101794906540","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.39379423279650999816866482130912060111","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.10519322535702571987089314750289501577","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82587763533601760002619014353812074700","series":"1.2.826.0.1.3680043.8.498.78463328924956608839864447259723713743","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82731606763525898032678680051515732914","series":"1.2.826.0.1.3680043.8.498.85670530960784589676539557074836136336","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.13294156143512360790250204091868306638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.78589980378304785460183858082202673238","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83000639623679673469640458882929412717","series":"1.2.826.0.1.3680043.8.498.89362389282240595724350466654433776041","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362","series":"1.2.826.0.1.3680043.8.498.21222797638450643734214834628226706468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097","series":"1.2.826.0.1.3680043.8.498.61467992414529262817151071208585469848","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83121378128521985713113311344685112825","series":"1.2.826.0.1.3680043.8.498.63481814017671062977679875258610111672","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.35955931361877219512211675270324024511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83253778356746892201320443660102495790","series":"1.2.826.0.1.3680043.8.498.27116331186729444306434454746854926036","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190","series":"1.2.826.0.1.3680043.8.498.12091440013023722724095109368584932459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.51723234111330646361299499106818832126","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83569065014916050100714967945484998462","series":"1.2.826.0.1.3680043.8.498.33035205420990272141669016507007177795","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","series":"1.2.826.0.1.3680043.8.498.96769371274727863051449574629461732309","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83847285247209768917850605454140194088","series":"1.2.826.0.1.3680043.8.498.85645744391629295560527180882784143625","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84007436672323587547274248257117331532","series":"1.2.826.0.1.3680043.8.498.10588546498448668497349091846024917497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.67243005773642910561381676401485772751","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.33988820173487867198021195437304886517","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.67922878155857332760526653763247213801","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84292569246911697141902796190329671261","series":"1.2.826.0.1.3680043.8.498.68284641334682068423589624395177717566","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655","series":"1.2.826.0.1.3680043.8.498.13403058894374843602524457326165800367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84486787871353595019134028899317143475","series":"1.2.826.0.1.3680043.8.498.86819799480832826059064448723439145740","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.96992552788137913424196388734180263732","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84709539128629951937294117586370340042","series":"1.2.826.0.1.3680043.8.498.12572744040497033642505542238269681881","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.26696639417174985416830777897733214078","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84857368518703570476541203284655412396","series":"1.2.826.0.1.3680043.8.498.31266754394232630758755039677470285931","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.64039713784524893908271862493017594320","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.12724074387958978914415622504920139493","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.72405248625771380966902743141599346635","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058","series":"1.2.826.0.1.3680043.8.498.81152662310963280716551735086083498735","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047","series":"1.2.826.0.1.3680043.8.498.19325728477200932976048483357015706093","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.75932046415951147005290581001780538812","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.11114464043462381774640824770025243641","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85431902914673414035492907270636012892","series":"1.2.826.0.1.3680043.8.498.83181806601721209187815098950655972147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.45975636922655757337013762899719593975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85545102350084131396266710667808696561","series":"1.2.826.0.1.3680043.8.498.11811577071107894011284784025545897855","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.74811637036319580301498936311263388357","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85649172620970194090967653766415467836","series":"1.2.826.0.1.3680043.8.498.82285863451881504999702430148345687987","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203","series":"1.2.826.0.1.3680043.8.498.36615115838631106759314689050467677451","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164","series":"1.2.826.0.1.3680043.8.498.15564555997210963302761715061258517615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85748882309364598407865143890978767674","series":"1.2.826.0.1.3680043.8.498.99209622434210007665938056867612098334","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.56476586152889649918753901981217578832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85882078780717905717989373451578844863","series":"1.2.826.0.1.3680043.8.498.57312018519229090936756940697874532790","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707","series":"1.2.826.0.1.3680043.8.498.15488894368812144330825666873565390636","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.10791776198099392739937291730764515350","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943","series":"1.2.826.0.1.3680043.8.498.74420387692374163639904877765284804255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86029947807373604003071651721618330827","series":"1.2.826.0.1.3680043.8.498.12899964489322375192154720102838193374","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86082522186055358984181640226529351801","series":"1.2.826.0.1.3680043.8.498.60738295230064386300697612849091477891","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797","series":"1.2.826.0.1.3680043.8.498.11733311676402227236170723709167470407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.63849640567097953042541956026223411536","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.92901503935873281165353367145591755012","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86315835011470465506843590014408831373","series":"1.2.826.0.1.3680043.8.498.52212099609381404802343675396547535574","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.96766708321188571237175105177064512020","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86409878732405036646840908998274047391","series":"1.2.826.0.1.3680043.8.498.10957583261250581175418093982875987837","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.12728651296201656069197035989409323236","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86516555757334047240474191101770683543","series":"1.2.826.0.1.3680043.8.498.35885500666898662418874774213424152281","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.68719184451541301627185734927500394438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.84893856199152670720640102428832843871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869","series":"1.2.826.0.1.3680043.8.498.43165171982117982483286465694743386545","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86968600239724310678905311244945464037","series":"1.2.826.0.1.3680043.8.498.27587238380139471988613191391521109317","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.72097519137019095991491211348023629106","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.10388327625808159683201614533928519054","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715","series":"1.2.826.0.1.3680043.8.498.15092958164932684371829674301324490473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.12264454503993907113850119754941151902","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","series":"1.2.826.0.1.3680043.8.498.12787036048566293526722536946152871709","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.11500009810356170205611549501131923131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285","series":"1.2.826.0.1.3680043.8.498.53203932545242738757052894279719602255","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.87413924318793242918916384453755054267","series":"1.2.826.0.1.3680043.8.498.12354927237313191595410531492412577278","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","series":"1.2.826.0.1.3680043.8.498.12269825253981474308705371086263582024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87550037046735743504446944390018580426","series":"1.2.826.0.1.3680043.8.498.67073549406703197783747523382502256759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188","series":"1.2.826.0.1.3680043.8.498.10276856008820677973080156553343491896","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015","series":"1.2.826.0.1.3680043.8.498.35725993421598619540877438779091604363","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87742737485142445536023876862256284839","series":"1.2.826.0.1.3680043.8.498.50622431833598861830670428705173255308","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.94907137172573498893033435970479018638","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.15531741542871951806966838995551669185","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87900330305833722536121052961334699743","series":"1.2.826.0.1.3680043.8.498.88479115611575927120757037243233913582","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.13073743297029776575509232586255357471","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88074618626471004030644935418777151905","series":"1.2.826.0.1.3680043.8.498.90748258187827318940466571803245668808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","series":"1.2.826.0.1.3680043.8.498.42472570299011529945489063882714161036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894","series":"1.2.826.0.1.3680043.8.498.50191931812608080101949655870885221037","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88457076333783710706782784856816112502","series":"1.2.826.0.1.3680043.8.498.11865952846068432167572684713032053524","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88480827968231439729532202780922145132","series":"1.2.826.0.1.3680043.8.498.99485201781626960465166910972383150690","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88565389747258713872180859306907392080","series":"1.2.826.0.1.3680043.8.498.48204754230019311789665661073810174188","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.12765935602009751825170347251075723122","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88620225126852321191690360547272730103","series":"1.2.826.0.1.3680043.8.498.48857541578619241026758216784813245494","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.22857504149251491601656881042529884467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88687994744258163246685527273559741656","series":"1.2.826.0.1.3680043.8.498.23629329040419491912563830218346647066","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941","series":"1.2.826.0.1.3680043.8.498.48364414489890709031057586278682005225","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","series":"1.2.826.0.1.3680043.8.498.31769567963647953847688273714648714043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88845193477394064535233691626377077328","series":"1.2.826.0.1.3680043.8.498.57003972167205548728029464218626643540","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.73101642489973088627839197522516656389","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88962401400598535739494671683916272476","series":"1.2.826.0.1.3680043.8.498.39441967324820626114380679972467638500","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883","series":"1.2.826.0.1.3680043.8.498.30937062690624470179586165866375268133","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.46627730676458340138010877834662584019","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","series":"1.2.826.0.1.3680043.8.498.12075128310634706976764486687048392273","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89204062904097086723738217311755833877","series":"1.2.826.0.1.3680043.8.498.46029976613990903959725260524085010681","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.34209961107909519435383762795660114661","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89326347774248795083360112232993521530","series":"1.2.826.0.1.3680043.8.498.13215309278085496244309677246991550620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.11521481771571287148760541925973562155","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89416235692252764852783739140428958560","series":"1.2.826.0.1.3680043.8.498.14687984466393339406119483652600676388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.11290890895953047069970614419209781155","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.94344730193093118761394608977460474202","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89590880980421560369495600204866789646","series":"1.2.826.0.1.3680043.8.498.67611298231637908727513625216922003388","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475","series":"1.2.826.0.1.3680043.8.498.19615881698825125088621174507081844236","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89842390998700046088751402253903762827","series":"1.2.826.0.1.3680043.8.498.28414348392018103388950625444790903857","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987","series":"1.2.826.0.1.3680043.8.498.93610072291334041644256048636227713769","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89967772422936726289461201791871771011","series":"1.2.826.0.1.3680043.8.498.25571603344853493775106780024920026058","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.12733336883536945170543416510359409275","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90103210546658057701715546323552886913","series":"1.2.826.0.1.3680043.8.498.24637788835416414363939922784546488730","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.32947317900404253628349635544686299367","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030","series":"1.2.826.0.1.3680043.8.498.11442584281422235612759614942557804477","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628","series":"1.2.826.0.1.3680043.8.498.69062709935665785914630832155756621468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.78884196383571523211757964647970695315","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.68825013301983908544465847600886327354","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768","series":"1.2.826.0.1.3680043.8.498.41066105973298540330359662549490653471","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90342120331229511020327052213821820086","series":"1.2.826.0.1.3680043.8.498.35365498214717111637261417961359460480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.53486479948817977088903787215813293430","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90490891782778285565462083536468192410","series":"1.2.826.0.1.3680043.8.498.58847507987572114764821322724913659976","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085","series":"1.2.826.0.1.3680043.8.498.31786692730518988844113857444892504420","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","series":"1.2.826.0.1.3680043.8.498.22370541352396355856280661499660772072","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90679199017361780574384378857945322240","series":"1.2.826.0.1.3680043.8.498.81408838250294002322132058982380335195","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90758414832541790868268922794274257839","series":"1.2.826.0.1.3680043.8.498.83054072780499203673047146985247596565","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90858397443821905121918628339486070979","series":"1.2.826.0.1.3680043.8.498.79639741846128585852300005985981995719","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90976227335921831531105845602655777296","series":"1.2.826.0.1.3680043.8.498.10147445232019973091373874316712623382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.36339182899386582499208201191032663488","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380","series":"1.2.826.0.1.3680043.8.498.78534418487392305011805415782176039294","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91139126799820974665417903552066103454","series":"1.2.826.0.1.3680043.8.498.87107878819082392079610782027147022109","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.77962513656195844097039752766981591995","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91237082716839283641165609666644466929","series":"1.2.826.0.1.3680043.8.498.78724002594231021408732381492120501174","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.32343999440300984799368550865935281357","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","series":"1.2.826.0.1.3680043.8.498.31071743258842759940030056875267780312","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042","series":"1.2.826.0.1.3680043.8.498.69810513871734635256892439443116413838","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.11181025571146209360935309996927283015","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.91499489440453244654238202136675083579","series":"1.2.826.0.1.3680043.8.498.82400365687980490747999895213491447253","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91611403539216247154562810076063878924","series":"1.2.826.0.1.3680043.8.498.11266937963725614921546729553121742165","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.44031822092276391321386048583051393978","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.11098209179837748851410526134564059294","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91883829404782805058767342767479130421","series":"1.2.826.0.1.3680043.8.498.12563976409811747257288482196325592747","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.10319532477084056049880319766537120357","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.11802077054554573809451910837364623558","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745","series":"1.2.826.0.1.3680043.8.498.35238773074537857898371376511743857464","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92126358259118823397932994812520542494","series":"1.2.826.0.1.3680043.8.498.12740237429980297594826142966106138743","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881","series":"1.2.826.0.1.3680043.8.498.16350803349713034213756218309096588284","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.35381246584622156114980955077720060488","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.86877948469721308568230235380468533213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654","series":"1.2.826.0.1.3680043.8.498.20921035338239794744061828661999430452","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92455257597708674644536154732642099964","series":"1.2.826.0.1.3680043.8.498.16140878781500950046208391210783243381","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92602515666185999130675702469977883616","series":"1.2.826.0.1.3680043.8.498.11489584221246893603110740314867605171","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917","series":"1.2.826.0.1.3680043.8.498.16688325247763141714657766500016213324","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","series":"1.2.826.0.1.3680043.8.498.38102042273002437963039098351217808688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.12835066277967830810822302073025078443","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.12871977925779305440243735750930061290","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92925901047699000727725798824992891055","series":"1.2.826.0.1.3680043.8.498.71878522925378587654321931757889459594","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142","series":"1.2.826.0.1.3680043.8.498.50232503849667967692138756098068751878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.30784763969947571881928974820469235728","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93182602591865019295859776146674822677","series":"1.2.826.0.1.3680043.8.498.11182957363144232096448589773154749776","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93222021909191248270818335069738080963","series":"1.2.826.0.1.3680043.8.498.94882443657121712579700999728004173336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.36582440717851974272988003929681321065","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209","series":"1.2.826.0.1.3680043.8.498.10145000165214049002600230371702128964","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93478919245462741704531324128061533679","series":"1.2.826.0.1.3680043.8.498.56513509043748142409976562923986909327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.71870238318207165071849293437988989788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487","series":"1.2.826.0.1.3680043.8.498.14481166903148757012715934335688559194","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.90362342307956700301419310070188671827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93848205752465320137433228658870667418","series":"1.2.826.0.1.3680043.8.498.24761391439706273721309498897665032389","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.55548680817825280622707232494129590297","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94002633501154835018549300667479350401","series":"1.2.826.0.1.3680043.8.498.18807015785231287979898742161747900498","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.49696709230828626962185950952615181470","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.13030645662072203667417280160544282779","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541","series":"1.2.826.0.1.3680043.8.498.11837805769268556775505066284113141849","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.94328216642444410827846344286227627425","series":"1.2.826.0.1.3680043.8.498.84420115376825978347075234927792475548","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.15576773826372332857837066554023005101","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402","series":"1.2.826.0.1.3680043.8.498.67939761478106677469103631979610028368","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94590838906158914501563687877014778914","series":"1.2.826.0.1.3680043.8.498.10680239557451517008053471370329561924","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.13371221246648032930709829204000564492","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94635110520016467882028406590905441740","series":"1.2.826.0.1.3680043.8.498.20608239686891480578583846951659451096","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.84909375845557144373437772354518005687","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94747403767627336827372096100476526821","series":"1.2.826.0.1.3680043.8.498.15948882860391265334553633455338718725","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.13335202991842748692366032759124750149","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.94822220672465914426995315295921716764","series":"1.2.826.0.1.3680043.8.498.92357246629041917188951574293008348157","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325","series":"1.2.826.0.1.3680043.8.498.88229990936916913406551818291012329483","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94955017957653373798729896600270551224","series":"1.2.826.0.1.3680043.8.498.19900140821141036991121571812512963035","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.40338185896521020279260308270107117081","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95091624999382167943525022414631622532","series":"1.2.826.0.1.3680043.8.498.19868726934075737565640884939986466010","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732","series":"1.2.826.0.1.3680043.8.498.57852970687720456137113031554807262978","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512","series":"1.2.826.0.1.3680043.8.498.91747733848784839000211626372241058800","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.19585656001809081617848552871539270213","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95334149193711661424105715330320414755","series":"1.2.826.0.1.3680043.8.498.82889611098294347698306132219305455173","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95376417067109020835342111882595126109","series":"1.2.826.0.1.3680043.8.498.13181678008625403433285369618017095108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95490888360439852685783476266753304657","series":"1.2.826.0.1.3680043.8.498.47222452656072814082307594088031386016","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.74235901085737471121068243976754633455","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.86287141996428963819045770863817988955","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95588576819758266413765165887081039863","series":"1.2.826.0.1.3680043.8.498.71987868866060974179918257972011728554","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.38528785427887659468677056088223009466","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.11971455229233006566392619173598725688","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.11841395465807846192188377788880507756","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95777611643095956714301310992151723603","series":"1.2.826.0.1.3680043.8.498.16038422738703098562775285351521853548","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608","series":"1.2.826.0.1.3680043.8.498.12313540521180451859386794041512874345","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95912798959818676102590029534562878721","series":"1.2.826.0.1.3680043.8.498.10035118823518592753987726109773657802","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96051765028927322365509303222455708654","series":"1.2.826.0.1.3680043.8.498.49960744130960595446625388097526833309","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","series":"1.2.826.0.1.3680043.8.498.43606849223736326058470933843217475476","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.97458730069669948614385896121207290508","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96174494699876332875347380499122120925","series":"1.2.826.0.1.3680043.8.498.10512020846677675137396983326050256950","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142","series":"1.2.826.0.1.3680043.8.498.60994568894234472642718936201403054271","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96448229695053562621553132798380456420","series":"1.2.826.0.1.3680043.8.498.11899829239919162077380502856443304875","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.35868244319891611779838424223571396954","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217","series":"1.2.826.0.1.3680043.8.498.11238828714312219567804876021078826766","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96587029379314618196381307882235433083","series":"1.2.826.0.1.3680043.8.498.93708708835451841177483184780917113759","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96657465815875587552665111166265086308","series":"1.2.826.0.1.3680043.8.498.74670715416582308069074200368036375696","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501","series":"1.2.826.0.1.3680043.8.498.18665293159434665446489669846530301823","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96943946564115188893954628924857155670","series":"1.2.826.0.1.3680043.8.498.50814910290590350596337772983494519669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.31161328675424343697301940527076363868","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97004483591331502283638578516018997228","series":"1.2.826.0.1.3680043.8.498.11750741404902153099611392545903076694","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.58129478820433524850741494874628695066","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97107472223824971578324157203273471706","series":"1.2.826.0.1.3680043.8.498.12315475229150265212576423408888578018","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.29782639754905522453285295875787544867","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97248397528214121944398119979516268902","series":"1.2.826.0.1.3680043.8.498.59267093191174207879034244788553385376","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","series":"1.2.826.0.1.3680043.8.498.12700785172962484122973859930068605140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97296323335289879687414244774227160450","series":"1.2.826.0.1.3680043.8.498.10573353590386776320858471679766400227","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606","series":"1.2.826.0.1.3680043.8.498.41894926106763201763076535288006146466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.40264716163351772539669515418810685749","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.42573750041245772853324546806980795594","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.67481737394483300868610391791476990225","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97564564380731258764291146761742841527","series":"1.2.826.0.1.3680043.8.498.12779735820241684524084980174035673960","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829","series":"1.2.826.0.1.3680043.8.498.68525949475482840953022307463227096001","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.10107135759922002075616133541624734496","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.88316353346155255497535951831671622986","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97797693721516872008302328638374252668","series":"1.2.826.0.1.3680043.8.498.21085078995289117222584265534599294589","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075","series":"1.2.826.0.1.3680043.8.498.57181301891002750823088111672467378060","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.97964867333695657200856428936254426086","series":"1.2.826.0.1.3680043.8.498.71441194083975153512434591655931403337","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.52207888705798655322160808942688765889","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.20375159001261904582329827106763725530","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98102475310924339032647376491442498974","series":"1.2.826.0.1.3680043.8.498.12011334525084157133095244936715573506","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676","series":"1.2.826.0.1.3680043.8.498.19257759129347987163756572975893031399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98151095279541278201351784048265686573","series":"1.2.826.0.1.3680043.8.498.91963478824768702319618334153956929993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720","series":"1.2.826.0.1.3680043.8.498.32551291137128246573206327843789972327","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98333658788330318335997032513277632526","series":"1.2.826.0.1.3680043.8.498.96662542884460241687380374571516918826","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635","series":"1.2.826.0.1.3680043.8.498.12958303001014931736766539613146148297","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.45663655790914232792492515944786863965","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98599322992880180540967962062521615070","series":"1.2.826.0.1.3680043.8.498.55180379562965045627703147704124349748","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.68258212163230318826249877827929918528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233","series":"1.2.826.0.1.3680043.8.498.20441887128385420203731491516073887255","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98871650330162988912501872031191530068","series":"1.2.826.0.1.3680043.8.498.93372874090607611724332437955666101409","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.12334618734697932701924489365527043348","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99052905406751383674032587504235261874","series":"1.2.826.0.1.3680043.8.498.14698492015671127569100829493669422055","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.29535045117330138064146771451823024103","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99178010550062268668346041362629896201","series":"1.2.826.0.1.3680043.8.498.14483630655722881520771178183454442331","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759","series":"1.2.826.0.1.3680043.8.498.71474897145521158177805585904341402752","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99309981642816644502709148844743995902","series":"1.2.826.0.1.3680043.8.498.11787479686216007522970084237151694286","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.11556975429759304805676039582571310285","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","series":"1.2.826.0.1.3680043.8.498.89615587694524573214449127869029607975","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99458163491875066451936719024359566928","series":"1.2.826.0.1.3680043.8.498.64254161195271916399063790656908648567","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.22783992299841288994316324352352315291","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99635784824711754116112241806442235135","series":"1.2.826.0.1.3680043.8.498.13335401444962951734014798664890513332","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263","series":"1.2.826.0.1.3680043.8.498.47921898712579242800383835611036857795","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99726731706411379395165830480890405115","series":"1.2.826.0.1.3680043.8.498.10731060894454049508761911820195549305","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820","series":"1.2.826.0.1.3680043.8.498.42192736836655935384244169150154303991","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262","series":"1.2.826.0.1.3680043.8.498.65124381355502126290104161253708117905","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99881620336512626242732899010111041290","series":"1.2.826.0.1.3680043.8.498.54144430573783949862972891646514764307","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.62870298129639401344127673739505626530","plane":"Coronal","fluid":1,"slot":"COR_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
